# 🗺️ 32 · Segmenting the cellular world: foundation models, diffusion features and topology
### One click, zero labels, correct shape: how the Xu Lab segments membranes and molecules in whole tomograms

**BiologyToResearch** · Part G — Frontiers, biology, research practice · Module 32 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/32_segmenting_the_cellular_world.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 2–3 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, scikit-image, scikit-learn, PyTorch (tiny), Matplotlib |
| **Prerequisites** | Module 17 (segmentation, Dice/IoU), Module 31 (picking with sparse labels); helpful: Modules 07 (missing wedge), 11 (picking metrics), 22 (clustering) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | CryoSAM (Zhao *et al.*, MICCAI 2024); unsupervised multi-scale segmentation with Stable Diffusion (Uddin *et al.*, CVPR 2026); TopoFuse (Salla *et al.*, ECCV 2026); MVGFormer (Li *et al.*, KBS 2025 — citation only) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments here are small, controlled *toy versions* of published methods. The real methods use huge pretrained
> "foundation models" (SAM, DINOv2, Stable Diffusion) that need gigabytes of weights and a GPU; we replace them with
> tiny hand-made stand-ins that keep the **algorithmic structure** of each paper. Our numbers therefore illustrate the
> core ideas; they do **not** reproduce the papers' results. **Nothing is stored in the repository:** molecules are
> real Protein Data Bank (PDB) models, downloaded the first time a notebook needs them and cached on your computer
> (offline, code-built stand-ins of the same size and symmetry are used instead); all images and data are simulated by
> code.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** in plain words what the foundation models SAM, DINOv2 and Stable Diffusion are, and why a lab would reuse
   them for cryo-ET instead of training from scratch.
2. **Implement** a toy version of CryoSAM — cross-plane self-prompting, masked-average query features, coarse-to-fine
   similarity matching with pruning, and 3D non-maximum suppression — and **measure** the precision, recall and F1 of
   "one-click" particle finding in a simulated cell.
3. **Compute** the leading eigenvectors of an attention-like affinity matrix in two ways (linear algebra, and the
   loss used by the Stable-Diffusion segmentation paper) and **use** them to separate membranes from particles with a
   circularity rule, with zero labels.
4. **Compute** Betti numbers (β₀, β₁, β₂) of 2D/3D masks and **show** that a segmentation can have a high Dice score yet
   the wrong topology.
5. **Explain** persistent homology as "features that survive while a threshold is lowered", and **implement** a sparse
   TopoFuse-style topology repair that reports a certificate (how many pixels it edited).
6. **Explain** tri-planar (xy/xz/yz) fusion and what it can and cannot fix under a missing wedge.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | check the computer, import tools, load the course toolkit | 5 min |
| 1 · Foundation models in one page | SAM, DINOv2, Stable Diffusion — what they are, what they give us | 10 min |
| 2 · A cell to segment | build a crowded in silico cell with a membrane vesicle | 10 min |
| 3 · CryoSAM-lite | one click → all ribosomes: self-prompting, feature matching, NMS, evaluation | 45 min |
| 4 · Segmentation from attention eigenvectors | spectral clustering of an attention matrix, membrane vs particle, pseudo-labels | 35 min |
| 5 · Topology | Betti numbers, persistence, a repair with a certificate, tri-planar fusion | 35 min |
| 6 · The big picture | how much supervision each method needs | 5 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A cellular tomogram is a 3D photograph of a living cell's interior: membranes wrap organelles and vesicles, filaments
crisscross the cytoplasm, and thousands of molecular machines float in between. Before any biology can be measured —
*how curved is this membrane? how many ribosomes sit on the endoplasmic reticulum? is this mitochondrion intact?* — every
voxel must be assigned to a structure. This is **segmentation**, and for decades it was done by hand, slice by slice,
taking days per tomogram.

Supervised deep networks (Module 17) need thousands of hand-painted voxels per structure — exactly what is missing for
a new organism or a new microscope. This module follows three recent Xu-lab answers: **borrow** a model pretrained on
millions of natural images and prompt it with *one click* (CryoSAM); **listen** to the internal attention of an image
generator to segment with *zero labels* (Stable-Diffusion segmentation); and **insist on the right shape** — membranes
that do not break, vesicles that stay closed — by measuring and repairing *topology* (TopoFuse).

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell checks what computer you are on.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet (the next cell installs it).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("skimage", "scikit-image"), ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from scipy import ndimage
from scipy.sparse.linalg import eigsh
from skimage.feature import peak_local_max
from skimage.filters import threshold_local, threshold_otsu
from skimage.graph import route_through_array
from skimage.measure import label as sk_label
from skimage.measure import regionprops
from skimage.morphology import skeletonize
from skimage.segmentation import watershed
from sklearn.linear_model import LogisticRegression

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # our tiny optimisation runs fine on a CPU
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| seed", SEED, "| device", DEVICE)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py):
building molecular-zoo maps, rotating volumes, packing a crowded cell (Module 09), simulating a tomogram with a
missing wedge and noise (Modules 06–09), and matching predicted to true particle positions (Module 11). Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_EXTRA_ENTRIES = {"5a1a": dict(name="β-galactosidase", organism="E. coli", mass_kDa=465, symmetry="D2")}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
def standin_atoms(pid):
    """Atomic model (dict of arrays) of the code-built stand-in for zoo entry `pid` (Å, centred at the origin).

    Used when the real entry cannot be downloaded: ideal α-helices arranged with the size, mass and symmetry of the
    PDB entry. Deterministic (the random numbers are seeded by the entry's name)."""
    pid = pid.lower()
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    return atoms


def pdb_cache_dir():
    """Folder (outside the repository) where downloaded PDB files are kept."""
    return os.environ.get("BIO2RESEARCH_PDB_CACHE") or os.path.join(os.path.expanduser("~"), ".cache",
                                                                    "biology_to_research", "pdb")


def _pdb_urls(pid, kind):
    """Download addresses of one PDB file: kind = 'entry' or 'assembly1' (mmCIF) or 'pdb' (legacy PDB format)."""
    pid = pid.lower()
    mid = pid[1:3]
    if kind == "entry":
        return [f"https://files.rcsb.org/download/{pid.upper()}.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/mmCIF/{mid}/{pid}.cif.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/{pid}.cif"]
    if kind == "assembly1":
        return [f"https://files.rcsb.org/download/{pid.upper()}-assembly1.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/assemblies/mmCIF/divided/{mid}/{pid}-assembly1.cif.gz"]
    if kind == "pdb":
        return [f"https://files.rcsb.org/download/{pid.upper()}.pdb.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/pdb/{mid}/pdb{pid}.ent.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{pid}.ent"]
    raise ValueError(kind)


def fetch_pdb(pid, kind="entry", timeout=60):
    """Text of a PDB file, downloaded once and cached outside the repository. Returns (text, where_from), or
    (None, reason) when no mirror can be reached (or BIO2RESEARCH_STANDINS is set)."""
    import gzip
    import urllib.request
    name = {"entry": f"{pid.lower()}.cif.gz", "assembly1": f"{pid.lower()}-assembly1.cif.gz",
            "pdb": f"{pid.lower()}.pdb.gz"}[kind]
    path = os.path.join(pdb_cache_dir(), name)
    if os.path.exists(path):
        return gzip.decompress(open(path, "rb").read()).decode("utf-8", "replace"), f"cached copy ({path})"
    if os.environ.get("BIO2RESEARCH_STANDINS"):
        return None, "BIO2RESEARCH_STANDINS is set"
    reason = "no mirror tried"
    for url in _pdb_urls(pid, kind):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                raw = response.read()
            text = (gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw).decode("utf-8", "replace")
            if ("_atom_site." not in text) and ("\nATOM  " not in text):
                raise ValueError("no atoms in the downloaded file")
            os.makedirs(pdb_cache_dir(), exist_ok=True)
            with open(path, "wb") as f:
                f.write(gzip.compress(text.encode()))
            return text, url
        except Exception as e:  # noqa: BLE001 — any failure: try the next mirror
            reason = f"{type(e).__name__}: {str(e)[:120]}"
    return None, reason


def _cif_tokens(line):
    """Split one line of an mmCIF loop into values (quoted values may contain spaces)."""
    if "'" not in line and '"' not in line:
        return line.split()
    import re
    toks = re.findall(r"'(?:[^']|'(?=\S))*'|\"(?:[^\"]|\"(?=\S))*\"|\S+", line)
    return [t[1:-1] if len(t) > 1 and t[0] == t[-1] and t[0] in "'\"" else t for t in toks]


_WATER = {"HOH", "DOD", "WAT", "H2O"}
def read_mmcif_atoms(text):
    """Atoms of the first model in an mmCIF file → dict(xyz, name, element, resname, chain, resnum).

    Reads the `_atom_site` loop; skips water and hydrogen, and keeps only the first of alternative atom locations."""
    lines = text.splitlines()
    i = next(k for k, line in enumerate(lines) if line.startswith("_atom_site."))
    cols = []
    while i < len(lines) and lines[i].startswith("_atom_site."):
        cols.append(lines[i].split()[0][len("_atom_site."):])
        i += 1
    c = {name: k for k, name in enumerate(cols)}
    get = lambda *names: next((c[n] for n in names if n in c), None)   # noqa: E731
    ix, iy, iz = c["Cartn_x"], c["Cartn_y"], c["Cartn_z"]
    i_el, i_name = get("type_symbol"), get("auth_atom_id", "label_atom_id")
    i_res, i_ch = get("auth_comp_id", "label_comp_id"), get("auth_asym_id", "label_asym_id")
    i_num, i_alt, i_mod = get("auth_seq_id", "label_seq_id"), get("label_alt_id"), get("pdbx_PDB_model_num")
    xyz, name, element, resname, chain, resnum = [], [], [], [], [], []
    first_model, buf = None, []
    for line in lines[i:]:
        if not line or line.startswith(("#", "_", "loop_", "data_")):
            break
        buf += _cif_tokens(line)
        while len(buf) >= len(cols):
            row, buf = buf[:len(cols)], buf[len(cols):]
            if i_mod is not None:
                first_model = row[i_mod] if first_model is None else first_model
                if row[i_mod] != first_model:
                    continue
            el = row[i_el].upper() if i_el is not None else row[i_name][:1].upper()
            if el in ("H", "D") or row[i_res] in _WATER:
                continue
            if i_alt is not None and row[i_alt] not in (".", "?", "A", "1"):
                continue
            xyz.append((float(row[ix]), float(row[iy]), float(row[iz])))
            name.append(row[i_name])
            element.append(el.capitalize() if len(el) > 1 else el)
            resname.append(row[i_res])
            chain.append(row[i_ch])
            num = row[i_num] if i_num is not None else "0"
            resnum.append(int(num) if num.lstrip("-").isdigit() else 0)
    return dict(xyz=np.array(xyz, np.float32), name=np.array(name), element=np.array(element),
                resname=np.array(resname), chain=np.array(chain), resnum=np.array(resnum))


_ATOMIC_MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "P": 30.974, "S": 32.06, "Se": 78.97, "Mg": 24.305,
                "Zn": 65.38, "Fe": 55.845, "Mn": 54.938, "K": 39.098, "Na": 22.99, "Cl": 35.45, "Ca": 40.078}
def model_mass_kDa(atoms):
    """Mass of an atomic model without hydrogens, plus ~8 % for the missing hydrogens (kDa)."""
    return 1.08 * sum(_ATOMIC_MASS.get(e, 12.0) for e in atoms["element"]) / 1000.0


def canonical_frame(atoms):
    """Centre a model and rotate it onto its principal axes — longest extent along x, shortest along z — with the
    signs fixed by the skewness of the atom distribution. A proteasome barrel then lies along x and a ring
    (RuBisCO, p97) lies in the x–y plane, like the stand-ins."""
    xyz = atoms["xyz"].astype(np.float64)
    X = xyz - xyz.mean(0)
    evecs = np.linalg.eigh(np.cov(X.T))[1][:, ::-1]                 # columns: principal axes, largest first
    Y = X @ evecs
    sign = np.sign(np.sum(Y ** 3, axis=0))
    sign[sign == 0] = 1.0
    Y *= sign
    if np.linalg.det(evecs * sign) < 0:                             # keep a proper rotation (no mirror image)
        Y[:, 2] *= -1
    out = dict(atoms)
    out["xyz"] = Y.astype(np.float32)
    return out


def pdb_atoms(pid, verbose=False):
    """Real atomic model of a PDB entry, in the canonical frame. Returns (atoms, description), or (None, reason).

    Uses the deposited entry unless its mass is far from ZOO's reference mass, in which case the first biological
    assembly is tried and the closer of the two kept (e.g. one ribosome rather than the two of a crystal)."""
    pid = pid.lower()
    ref = ZOO.get(pid, {}).get("mass_kDa") or _EXTRA_ENTRIES.get(pid, {}).get("mass_kDa")
    candidates, reason = [], ""
    for kind in ("entry", "assembly1"):
        if candidates and (ref is None or abs(np.log(model_mass_kDa(candidates[0][0]) / ref)) < np.log(1.3)):
            break
        text, where = fetch_pdb(pid, kind)
        if text is None:
            reason = where
            continue
        try:
            atoms = read_mmcif_atoms(text)
        except (StopIteration, KeyError, ValueError) as e:
            reason = f"could not read the file ({type(e).__name__})"
            continue
        if len(atoms["xyz"]):
            label = "deposited entry" if kind == "entry" else "biological assembly 1"
            candidates.append((atoms, f"PDB {pid.upper()} ({label}; {len(atoms['xyz']):,} atoms; from {where})"))
    if not candidates:
        return None, reason
    atoms, desc = min(candidates, key=lambda a: abs(np.log(model_mass_kDa(a[0]) / ref)) if ref else 0.0)
    if verbose:
        print(desc)
    return canonical_frame(atoms), desc


_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays, Å, centred) of zoo entry `pid`: the real PDB model when it can be downloaded, the
    code-built stand-in otherwise. The source is recorded in `atoms_source(pid)`."""
    pid = pid.lower()
    if pid not in _ATOMS_CACHE:
        atoms, desc = pdb_atoms(pid)
        if atoms is None:
            atoms, desc = standin_atoms(pid), f"code-built stand-in modelled on PDB {pid.upper()} (download failed: {desc})"
        _ATOMS_CACHE[pid] = (atoms, desc)
    return _ATOMS_CACHE[pid][0]


def atoms_source(pid):
    """Where the atoms of zoo entry `pid` came from (a real PDB file or the offline stand-in)."""
    zoo_atoms(pid)
    return _ATOMS_CACHE[pid.lower()][1]


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0, "Mg": 12.0, "Zn": 30.0, "Fe": 26.0, "K": 19.0,
      "Na": 11.0, "Cl": 17.0, "Ca": 20.0, "Mn": 25.0, "Se": 34.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of a zoo entry, painted from its atoms (`zoo_atoms`)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, atoms_source(pid)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def electron_wavelength_A(kV: float = 300.0) -> float:
    """Relativistic electron wavelength in Å (0.0197 Å at 300 kV)."""
    V = kV * 1e3
    return 12.2643 / np.sqrt(V * (1 + 0.978466e-6 * V))


def ctf_2d(shape, pixel_A: float, defocus_um: float = 3.0, kV: float = 300.0,
           Cs_mm: float = 2.7, amp_contrast: float = 0.07, B: float = 0.0) -> np.ndarray:
    """Contrast transfer function on an unshifted FFT grid (underfocus is positive).

    CTF(k) = −[√(1−A²)·sin χ(k) + A·cos χ(k)]·exp(−B k²/4),
    χ(k) = π λ Δf k² − (π/2) C_s λ³ k⁴, with k in 1/Å.
    """
    lam = electron_wavelength_A(kV)
    k2 = sum(kk**2 for kk in freq_grid(shape)) / pixel_A**2
    dz = defocus_um * 1e4
    cs = Cs_mm * 1e7
    chi = np.pi * lam * dz * k2 - 0.5 * np.pi * cs * lam**3 * k2**2
    ctf = -(np.sqrt(1 - amp_contrast**2) * np.sin(chi) + amp_contrast * np.cos(chi))
    return (ctf * np.exp(-B * k2 / 4)).astype(np.float32)


def match_points(pred, true, radius: float):
    """Greedy one-to-one matching of predicted to true coordinates within ``radius``.

    Returns dict(precision, recall, f1, tp, fp, fn).
    """
    true = np.asarray(true, float)
    pred = np.asarray(pred, float).reshape(-1, true.shape[-1] if true.ndim == 2 else 3)
    if len(pred) == 0 or len(true) == 0:
        tp = 0
    else:
        d = np.linalg.norm(pred[:, None, :] - true[None, :, :], axis=-1)
        used_p, used_t, tp = set(), set(), 0
        for i, j in zip(*np.unravel_index(np.argsort(d, axis=None), d.shape)):
            if d[i, j] > radius:
                break
            if i not in used_p and j not in used_t:
                used_p.add(i); used_t.add(j); tp += 1
    fp, fn = len(pred) - tp, len(true) - tp
    p = tp / max(len(pred), 1)
    r = tp / max(len(true), 1)
    return dict(precision=p, recall=r, f1=2 * p * r / max(p + r, 1e-12), tp=tp, fp=fp, fn=fn)


def support_radius(t: np.ndarray, level: float = 0.1) -> float:
    """Radius (voxels) of the smallest centred sphere containing all voxels above ``level``·max."""
    pts = np.argwhere(t > level * t.max())
    if len(pts) == 0:
        return t.shape[0] / 2
    c = (np.array(t.shape) - 1) / 2
    return float(np.linalg.norm(pts - c, axis=1).max()) + 1.0


def make_crowded_cell(templates: dict, counts: dict, shape=(48, 160, 160), rng=None,
                      membrane: bool = True, tries_per_particle: int = 400):
    """Pack randomly rotated templates into a volume without overlap (bounding spheres).

    A simple rejection-sampling stand-in for the packing used by Pei et al. (2016)
    and Liu et al. (2020).  Particles are placed in random class order so that every
    class gets a fair chance.  Returns (clean_volume, particles) where each particle
    is dict(cls, center_zyx, R).  Optionally adds a spherical membrane vesicle.
    """
    rng = rng or np.random.default_rng(0)
    vol = np.zeros(shape, np.float32)
    occ_c, occ_r, particles = [], [], []
    if membrane:  # a vesicle: thin shell, ~4 nm thick lipid bilayer at 1 nm/voxel
        zc, yc, xc = shape[0] / 2, shape[1] * 0.78, shape[2] * 0.22
        rad = min(shape[1], shape[2]) * 0.16
        g = np.meshgrid(*[np.arange(n) for n in shape], indexing="ij")
        d = np.sqrt((g[0] - zc)**2 + (g[1] - yc)**2 + (g[2] - xc)**2)
        vol += 0.6 * np.exp(-((d - rad)**2) / (2 * 1.5**2))
        occ_c.append(np.array([zc, yc, xc])); occ_r.append(rad + 3)
    radii = {cls: support_radius(templates[cls], level=0.35) for cls in counts}
    order = [cls for cls, n in counts.items() for _ in range(n)]
    rng.shuffle(order)
    for cls in order:
        t, rad_t = templates[cls], radii[cls]
        for _ in range(tries_per_particle):
            c = np.array([rng.uniform(rad_t, s - rad_t) if s > 2 * rad_t else s / 2 for s in shape])
            if any(np.linalg.norm(c - oc) < rad_t + orr for oc, orr in zip(occ_c, occ_r)):
                continue
            R = random_rotation(rng)
            sub = rotate_volume(t, R)
            h = np.array(sub.shape) // 2
            lo = np.round(c).astype(int) - h
            sl_v, sl_s = [], []
            for a in range(3):
                a0, a1 = max(lo[a], 0), min(lo[a] + sub.shape[a], shape[a])
                sl_v.append(slice(a0, a1)); sl_s.append(slice(a0 - lo[a], a1 - lo[a]))
            vol[tuple(sl_v)] += sub[tuple(sl_s)]
            occ_c.append(c); occ_r.append(rad_t)
            particles.append(dict(cls=cls, center_zyx=c, R=R))
            break
    return vol, particles


def simulate_tomogram(clean: np.ndarray, rng, snr: float = 0.05, tilt_max_deg: float = 60.0,
                      ctf_pixel_A: float | None = 10.0, defocus_um: float = 5.0) -> np.ndarray:
    """Fast tomogram simulation: (optional 3D CTF-like filter) → missing wedge → noise.

    The CTF is applied isotropically here (a simplification of per-tilt CTF).
    """
    v = clean
    if ctf_pixel_A:
        ctf = ctf_2d(clean.shape, ctf_pixel_A, defocus_um=defocus_um, kV=300, B=0)
        v = apply_fourier_mask(v, -ctf)          # sign flip: dense objects appear bright
    v = apply_fourier_mask(v, missing_wedge_mask(v.shape, tilt_max_deg))
    return add_noise_snr(v, snr, rng)
print("🧰 toolkit loaded")

---
## 📍 Where we are

In Module 17 you trained a network to segment subtomograms voxel by voxel with full labels, and in Module 31 you
picked particles from only ~10 labelled examples. This module removes the labels almost entirely: **one click**
(Section 3) or **none at all** (Section 4). It then asks a question that voxel-wise scores such as Dice ignore: is the
*shape* right — are membranes unbroken and vesicles closed (Section 5)?

---
## 1 · Foundation models in one page

A **foundation model** is a large neural network pretrained once, at enormous cost, on a huge and diverse dataset, and
then reused for many tasks it was never explicitly trained for. Three of them appear in the Xu-lab papers of this
module:

| Model (year) | Trained on | What it gives you | How cryo-ET work reuses it |
|---|---|---|---|
| **SAM** — *Segment Anything* (Meta AI, 2023) | ~11 million photos with ~1.1 billion masks | a **promptable segmenter**: click a point (or draw a box) on a 2D image → a mask of the object under the click, plus a confidence score | CryoSAM segments a particle slice by slice from one click; TopoFuse uses SAM's image encoder on xy, xz and yz slices |
| **DINOv2** (Meta AI, 2023) | ~142 million curated images, *self-supervised* (no labels) | a **feature vector for every 14 × 14-pixel patch**; patches showing the same kind of object get similar vectors | CryoSAM compares these features to find *other* particles that look like the clicked one |
| **Stable Diffusion** (2022) | billions of image–caption pairs | an **image generator** that removes noise step by step with a U-Net containing *self-attention* layers | the attention maps tell which pixels "belong together" → unsupervised segmentation (Uddin *et al.* 2026) |

Why bother? Because a network that has seen a billion object boundaries "knows" what an edge, a blob and a texture
are — knowledge that transfers surprisingly well even to grey, noisy tomograms, *without any cryo-ET labels*.

💡 **What we do instead (and why).** These models need several gigabytes of weights and ideally a GPU, so we build
faithful *miniatures*:

| Real component | Our toy stand-in | What is preserved |
|---|---|---|
| DINOv2 patch features | a **filter bank** (blur, blob detector, edge strength, orientation coherence) at 3 scales, computed slice by slice along z, y and x | "every voxel gets a descriptive vector; similar structures get similar vectors" |
| SAM | **region growing** from the clicked point on a smoothed slice | "a click on a 2D slice → a mask + a confidence map" |
| Stable Diffusion self-attention | **softmax(F Fᵀ/√d)** over patch features | "a row-normalised affinity matrix between all patches" |

Everything *around* these components — the propagation logic, matching, pruning, eigen-decomposition, circularity
split, topology repair — follows the papers and the lab's released code.

---
## 2 · A cell to segment

We reuse the course's in silico cell (Module 09): ribosomes (4V4R), 20S proteasomes (3J9I) and RuBisCO (1BXN) packed
into a 144 × 144 × 64 nm box with a membrane vesicle. CryoSAM was run on tomograms that had been CTF-corrected and
denoised (with the Warp software), so we simulate the missing wedge (±60°) and noise (SNR 0.1) **without** the CTF and
apply a light Gaussian denoising (σ = 1 voxel = 1 nm).

In [ ]:
# ── 2.1 Build the zoo and the in silico cell ──
t0 = time.time()
CAST = ["4v4r", "3j9i", "1bxn"]
templates = {}
for pid in CAST:
    raw, voxel_A, source = zoo_map(pid)
    templates[pid] = prepare_map(raw, size=32, sigma=1.0)
    print(f"  {pid.upper()} {ZOO[pid]['name']:18s} ← {source}")
counts = {"4v4r": 14, "3j9i": 12, "1bxn": 24}
clean, particles = make_crowded_cell(templates, counts, shape=(64, 144, 144), rng=np.random.default_rng(1))
tomo = simulate_tomogram(clean, np.random.default_rng(2), snr=0.1, tilt_max_deg=60, ctf_pixel_A=None)
den = normalize(ndimage.gaussian_filter(tomo, 1.0))          # light denoising; units = standard deviations
centers = np.array([p["center_zyx"] for p in particles])
classes = np.array([p["cls"] for p in particles])
print(f"placed {len(particles)} particles in {time.time() - t0:.1f} s:",
      {ZOO[c]["name"]: int((classes == c).sum()) for c in CAST})

In [ ]:
# ── 2.2 Ground-truth masks of individual particles (for scoring only) ──
def particle_mask(p: dict, templates: dict, shape, level: float = 0.35) -> np.ndarray:
    """Binary mask of one placed particle: its template above `level`·max, rotated and placed exactly as in the cell."""
    t = templates[p["cls"]]
    sub = rotate_volume(t, p["R"]) > level * t.max()
    out = np.zeros(shape, bool)
    lo = np.round(p["center_zyx"]).astype(int) - np.array(sub.shape) // 2
    sl_v, sl_s = [], []
    for a in range(3):
        a0, a1 = max(lo[a], 0), min(lo[a] + sub.shape[a], shape[a])
        sl_v.append(slice(a0, a1)); sl_s.append(slice(a0 - lo[a], a1 - lo[a]))
    out[tuple(sl_v)] = sub[tuple(sl_s)]
    return out


ribo_idx = np.where(classes == "4v4r")[0]
ribo_true = centers[ribo_idx]
gt_ribo_mask = np.zeros(den.shape, bool)
for i in ribo_idx:
    gt_ribo_mask |= particle_mask(particles[i], templates, den.shape)
print(f"{len(ribo_idx)} ribosomes occupy {gt_ribo_mask.sum():,} voxels ({100 * gt_ribo_mask.mean():.1f} % of the box)")

z_mid = 35
marker = {"4v4r": ("o", "tab:red"), "3j9i": ("s", "tab:blue"), "1bxn": ("^", "tab:green")}
fig, axes = plt.subplots(1, 2, figsize=(9, 4.4))
axes[0].imshow(clean[z_mid]); axes[0].set_title("true density (z = 35 nm)")
axes[1].imshow(den[z_mid]); axes[1].set_title("denoised tomogram slice (what we segment)")
for p in particles:
    z, y, x = p["center_zyx"]
    if abs(z - z_mid) < 8:
        m, c = marker[p["cls"]]
        axes[1].scatter(x, y, marker=m, facecolors="none", edgecolors=c, s=150, lw=1.3)
for pid, (m, c) in marker.items():
    axes[1].scatter([], [], marker=m, facecolors="none", edgecolors=c, label=ZOO[pid]["name"])
axes[1].legend(fontsize=7, loc="lower right")
for ax in axes:
    ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** We now have a tomogram with an answer key: the class, centre and orientation of every
particle, and a voxel mask for every ribosome. The vesicle (bottom left) is a *membrane* — a thin closed shell — which we
will meet again in Section 5. Our goal for Section 3: **click one ribosome and find all the others.**

---
## 3 · CryoSAM-lite: one click → every ribosome

CryoSAM (Zhao *et al.*, MICCAI 2024) is *training-free*: it never updates a single network weight. It chains two frozen
2D foundation models into a 3D pipeline:

```text
 click (z,y,x) ──► (1) SAM on the clicked slice ──► cross-plane self-prompting ──► 3D mask of the clicked particle
                                                                                        │
 tri-view DINOv2 features of the whole tomogram ◄──────────── masked average = query ◄─┘
       │
       └─► (2) hierarchical matching: cosine similarity at 1/16 → 1/8 → 1/4 resolution, prune < 0.5
                └─► 3D non-maximum suppression → new point prompts ──► back to (1) for every new particle
```

We build each block in turn.

### 3.1 Tri-view "foundation" features

DINOv2 is a 2D model, so CryoSAM runs it on every slice of the tomogram **three times** — on xy slices (stacked along z),
on xz slices and on yz slices — and concatenates the three results into a **tri-view feature volume**
$\mathbf F=[\mathbf Z,\mathbf Y,\mathbf X]\in\mathbb R^{D\times H\times W\times 3C}$. *In words:* every voxel gets three
descriptions, one from each viewing direction, glued together.

Our stand-in for DINOv2 is a classical **filter bank**, applied to each 2D slice (σ = 0 along the slicing axis):

| filter | formula (2D, scale σ) | what it detects |
|---|---|---|
| Gaussian blur | $G_\sigma * I$ | local brightness (dense matter) |
| Laplacian of Gaussian (LoG), scale-normalised | $\sigma^2\,\nabla^2 G_\sigma * I$ | blobs of radius ≈ $\sigma\sqrt2$ (negative at bright blob centres) |
| gradient magnitude | $\sigma\,\lvert\nabla (G_\sigma * I)\rvert$ | edges / boundaries |
| structure-tensor coherence | $\sqrt{(J_{11}-J_{22})^2+4J_{12}^2}\,/\,(J_{11}+J_{22})$ | 0 = no preferred direction (blob), 1 = a line or sheet (membrane) |

with $J_{ab}$ the smoothed products of the two in-plane derivatives. 4 filters × 3 scales (σ = 1, 3, 6 nm) × 3 views
= **36 numbers per voxel**.

⚠️ **One extra step our toy needs.** Our 36 filters are highly *correlated* — almost all of them shout "bright blob here!"
— so the cosine similarity between any two particles is ≈ 1 and a threshold of 0.5 would mean nothing. Real DINOv2
features are far more diverse. We therefore **decorrelate ("whiten")** the features with PCA: rotate them onto their
principal axes, keep the top 8 and give each unit variance. (This is our engineering fix, not part of CryoSAM.)

In [ ]:
# ── 3.1 Tri-view filter-bank features ──
def tri_view_features(vol: np.ndarray, scales=(1.0, 3.0, 6.0), truncate: float = 3.0):
    """Per-slice 2D filter bank (Gaussian, LoG, gradient magnitude, structure-tensor coherence) along z, y and x.

    All derivatives are taken *within* each slice (the two in-plane axes); σ = 0 along the slicing axis.
    Returns (features[D, H, W, 36] standardised per channel, list of channel names)."""
    F = np.empty(vol.shape + (12 * len(scales),), np.float16)   # half precision is plenty and saves memory
    names = []

    def store(i: int, x: np.ndarray):
        """Standardise one feature channel (zero mean, unit variance) and store it."""
        F[..., i] = (x - x.mean()) / (x.std() + 1e-6)
    for ax, view in [(0, "xy"), (1, "xz"), (2, "yz")]:
        a1, a2 = [a for a in range(3) if a != ax]              # the two in-plane axes of this view
        for s in scales:
            sig = [s, s, s]; sig[ax] = 0                       # σ = 0 along the slicing axis → 2D filtering per slice

            def gauss(order1: int = 0, order2: int = 0, img=vol, sigma=sig):
                """Gaussian (derivative) filter with the given derivative orders along the two in-plane axes."""
                order = [order1 if a == a1 else order2 if a == a2 else 0 for a in range(3)]
                return ndimage.gaussian_filter(img, sigma, order=order, truncate=truncate)
            g1, g2 = gauss(1, 0), gauss(0, 1)                   # in-plane first derivatives
            s2 = [2 * v for v in sig]                           # integration scale of the structure tensor
            j11, j22, j12 = (gauss(img=prod, sigma=s2) for prod in (g1 * g1, g2 * g2, g1 * g2))
            i = len(names)
            store(i, gauss())                                    # blur
            store(i + 1, (gauss(2, 0) + gauss(0, 2)) * s * s)    # scale-normalised Laplacian of Gaussian
            store(i + 2, np.sqrt(g1 ** 2 + g2 ** 2) * s)         # scale-normalised gradient magnitude
            store(i + 3, np.sqrt((j11 - j22) ** 2 + 4 * j12 ** 2) / (j11 + j22 + 1e-6))   # coherence
            names += [f"{view} blur σ{s:g}", f"{view} LoG σ{s:g}", f"{view} |grad| σ{s:g}", f"{view} coherence σ{s:g}"]
    return F, names


def whiten(F: np.ndarray, k: int = 8, step: int = 37, chunk: int = 200_000):
    """PCA-whiten features: project on the top-k principal axes and scale each to unit variance."""
    X = F.reshape(-1, F.shape[-1])
    C = np.cov(X[::step].astype(np.float32).T)                  # covariance from a subsample of voxels
    w, V = np.linalg.eigh(C)
    order = np.argsort(w)[::-1][:k]
    P = (V[:, order] / np.sqrt(w[order])).astype(np.float32)   # projection matrix (36 × k)
    Fw = np.empty((X.shape[0], k), np.float32)
    for a in range(0, X.shape[0], chunk):                       # project in chunks to keep memory low
        Fw[a:a + chunk] = X[a:a + chunk].astype(np.float32) @ P
    return Fw.reshape(F.shape[:3] + (k,)), w[order] / w.sum()


t0 = time.time()
F36, feat_names = tri_view_features(den)
Fw, explained = whiten(F36, k=8)
print(f"tri-view features {F36.shape} → whitened {Fw.shape} in {time.time() - t0:.1f} s")
print("variance explained by the 8 kept axes:", np.round(explained, 3), f"(total {explained.sum():.2f})")

In [ ]:
# ── 3.1b Look at the features ──
show = ["xy blur σ1", "xy LoG σ6", "xy |grad| σ1", "xy coherence σ3"]
fig, axes = plt.subplots(1, 5, figsize=(12, 2.8))
for ax, name in zip(axes, show):
    ax.imshow(F36[z_mid, :, :, feat_names.index(name)].astype(np.float32), cmap="viridis")
    ax.set_title(name, fontsize=8); ax.axis("off")
rgb = Fw[z_mid, :, :, :3].copy()
rgb = (rgb - np.percentile(rgb, 1, axis=(0, 1))) / (np.percentile(rgb, 99, axis=(0, 1)) - np.percentile(rgb, 1, axis=(0, 1)))
axes[4].imshow(np.clip(rgb, 0, 1)); axes[4].set_title("top-3 whitened axes as RGB", fontsize=8); axes[4].axis("off")
fig.suptitle(f"Filter-bank features on slice z = {z_mid} nm (our stand-in for DINOv2)", fontsize=10)
plt.tight_layout(); plt.show()
del F36                                                          # free ~200 MB; from now on we use the whitened Fw

🔍 **What just happened?** Each voxel now carries 8 decorrelated numbers summarising brightness, blobness at three
sizes, edges and "line-ness" seen from three directions. In the RGB panel, ribosomes, small particles, the membrane and
the background take on different colours — the same kind of picture people make from the first three principal
components of DINOv2 features.

### 3.2 A promptable segmenter ("SAM") and cross-plane self-prompting

Our toy SAM receives a 2D slice and a click $(y, x)$. It smooths the slice, keeps the pixels brighter than half the
brightness at the click, and returns the **connected region containing the click** (the mask $\mathbf M$) plus a
**confidence map** $\mathbf C$ (distance from the mask border × brightness, largest deep inside the object). Like SAM's
area filter in the CryoSAM code (masks larger than 32 × 32 pixels are rejected), we refuse masks larger than 900 pixels,
and like SAM's low score on empty background, we refuse clicks on voxels dimmer than 1.5 standard deviations.

**Cross-plane self-prompting** (CryoSAM's eqs.) then walks through the volume:

$$(\mathbf C_{z},\mathbf M_{z})=\mathrm{SAM}[\mathbf I_{z}\mid \text{click}],\qquad
\mathbf Q_{z}=\arg\max_{x,y}\mathbf C_{z},\qquad
(\mathbf C_{z+1},\mathbf M_{z+1})=\mathrm{SAM}[\mathbf I_{z+1}\mid \mathbf Q_{z}]$$

*In words:* segment the clicked slice, take its most confident point as the click for the next slice, and repeat —
until the new mask overlaps the previous one by less than $\tau_{IoU}=0.5$ (intersection over union), which means we
have walked off the particle. The paper runs this in ±z, ±y and ±x and takes the union; the released code walks only
along ±z. We implement both.

In [ ]:
# ── 3.2 Toy SAM and cross-plane self-prompting ──
OBJ_THR = 1.5                                                    # "is there an object here?" (in std units)
SMOOTHED = {ax: ndimage.gaussian_filter(den, [0 if a == ax else 1.0 for a in range(3)]) for ax in range(3)}


def toy_sam_2d(sm_slice: np.ndarray, point, rel: float = 0.5, max_area: int = 900):
    """Promptable 2D 'segmenter': connected region brighter than rel × (value at the click).

    Returns (mask, confidence) or (None, None) if the click is on background or the region is too large."""
    y, x = int(point[0]), int(point[1])
    seed_val = sm_slice[max(y - 1, 0):y + 2, max(x - 1, 0):x + 2].max()
    if seed_val < OBJ_THR:
        return None, None
    lab, _ = ndimage.label(sm_slice > rel * seed_val)
    if lab[y, x] == 0:
        return None, None
    mask = lab == lab[y, x]
    if mask.sum() > max_area:
        return None, None
    conf = ndimage.distance_transform_edt(mask) * sm_slice      # high deep inside a bright object
    return mask, conf


def iou(a: np.ndarray, b: np.ndarray) -> float:
    """Intersection over union of two boolean masks."""
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else 0.0


def self_prompt(prompt_zyx, axis: int = 0, tau_iou: float = 0.5, return_ious: bool = False):
    """Cross-plane self-prompting along one axis (both directions) starting from a 3D click."""
    v = np.moveaxis(SMOOTHED[axis], axis, 0)
    p = [int(round(c)) for c in prompt_zyx]
    s0 = p.pop(axis)
    out = np.zeros(v.shape, bool)
    m0, c0 = toy_sam_2d(v[s0], p)
    ious = {}
    if m0 is None:
        return (np.moveaxis(out, 0, axis), ious) if return_ious else np.moveaxis(out, 0, axis)
    out[s0] = m0
    for step in (+1, -1):
        prev, conf, s = m0, c0, s0
        while 0 <= s + step < v.shape[0]:
            q = np.unravel_index(np.argmax(conf), conf.shape)   # most confident point = next click
            m, c = toy_sam_2d(v[s + step], q)
            ious[s + step] = 0.0 if m is None else iou(m, prev)
            if m is None or ious[s + step] < tau_iou:
                break
            s += step; out[s] = m; prev, conf = m, c
    out = np.moveaxis(out, 0, axis)
    return (out, ious) if return_ious else out


def segment_3d(prompt_zyx, axes=(0, 1, 2)) -> np.ndarray:
    """Union of self-prompted masks along the requested axes (paper: all three; released code: z only)."""
    m = np.zeros(den.shape, bool)
    for ax in axes:
        m |= self_prompt(prompt_zyx, ax)
    return m


click = np.round(centers[ribo_idx[0]]).astype(int)              # the one ribosome we click on
gt_click = particle_mask(particles[ribo_idx[0]], templates, den.shape)
t0 = time.time()
mask_z, ious_z = self_prompt(click, axis=0, return_ious=True)
mask_3 = segment_3d(click)
print(f"click at (z, y, x) = {tuple(int(c) for c in click)} nm; segmentation took {time.time() - t0:.2f} s")
print(f"3D IoU with the true ribosome mask: z-only {iou(mask_z, gt_click):.2f} | union of z, y, x {iou(mask_3, gt_click):.2f}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 1, 1.3]})
m0, c0 = toy_sam_2d(SMOOTHED[0][click[0]], click[1:])
y0, x0 = max(click[1] - 22, 0), max(click[2] - 22, 0)
win = (slice(y0, y0 + 44), slice(x0, x0 + 44))                  # a 44 × 44 nm window around the click
axes[0].imshow(den[click[0]][win]); axes[0].contour(m0[win], [0.5], colors="tab:orange")
axes[0].plot(click[2] - x0, click[1] - y0, "r+", ms=12, mew=2); axes[0].set_title("click + toy-SAM mask"); axes[0].axis("off")
axes[1].imshow(c0[win], cmap="viridis"); q = np.unravel_index(np.argmax(c0[win]), c0[win].shape)
axes[1].plot(q[1], q[0], "wx", ms=10, mew=2); axes[1].set_title("confidence → next click (×)"); axes[1].axis("off")
zs = sorted(ious_z)
axes[2].plot(zs, [ious_z[z] for z in zs], "o-", ms=3)
axes[2].axhline(0.5, color="tab:red", ls="--", label="stop when IoU < 0.5")
axes[2].axvline(click[0], color="gray", ls=":", label="clicked slice")
axes[2].set_xlabel("slice z (nm)"); axes[2].set_ylabel("IoU with previous slice"); axes[2].legend(fontsize=7)
axes[2].set_title("propagation along ±z")
plt.tight_layout(); plt.show()

🔍 **What just happened?** One click produced a 3D mask of the clicked ribosome. Walking along z alone recovers part of
the particle; the union of the three walking directions (the paper's version) overlaps the truth better. The IoU curve
shows where propagation stopped: the masks of neighbouring slices suddenly stop overlapping when the walk leaves the
particle (or reaches its elongated, missing-wedge-smeared tips).

### 3.3 From one particle to all particles: the query and hierarchical matching

CryoSAM summarises the clicked particle(s) by the **masked average** of the features inside the mask(s):

$$\mathbf F_Q=\frac{\sum_i\sum_{zyx}\mathbf M^i_{zyx}\,\mathbf F_{zyx}}{\sum_i\|\mathbf M^i\|_0}$$

*In words:* average the feature vectors of all voxels inside the prompted masks ($\|\mathbf M\|_0$ = number of voxels in
the mask). (The released code instead samples the feature at the mask's **centroid**; we will compare both.) Every
voxel is then scored by **cosine similarity** to the query,

$$S_{zyx}=\frac{\mathbf F_Q\cdot\mathbf F_{zyx}}{\|\mathbf F_Q\|\,\|\mathbf F_{zyx}\|}\in[-1,1],$$

i.e. how well the voxel's feature vector points in the same direction as the query. Computing this for every voxel of a
real tomogram is slow, so CryoSAM goes **coarse to fine**: average-pool the features by 16, 8, then 4 (DINOv2 patches are
already coarse); at each level, discard voxels with $S<\tau_{sim}=0.5$ and only evaluate the survivors at the next finer
level. At the finest level, a **3D non-maximum suppression** (NMS: keep a voxel only if it is the maximum of its
neighbourhood; a 5³ max-pool in the code) turns the similarity map into point prompts. Our voxels are already 1 nm, so
our pyramid is 1/4 → 1/2 → 1, and our NMS window is sized to the clicked particle (its equivalent-sphere diameter).

In [ ]:
# ── 3.3 Query feature and coarse-to-fine matching ──
def downsample_features(F: np.ndarray, r: int) -> np.ndarray:
    """Average-pool a feature volume by an integer factor r along z, y and x."""
    if r == 1:
        return F
    D, H, W, C = F.shape
    return F[:D // r * r, :H // r * r, :W // r * r].reshape(D // r, r, H // r, r, W // r, r, C).mean((1, 3, 5))


def cosine_to(F: np.ndarray, q: np.ndarray) -> np.ndarray:
    """Cosine similarity between every feature vector in F (…, C) and the query q (C,)."""
    return (F @ (q / np.linalg.norm(q))) / (np.linalg.norm(F, axis=-1) + 1e-6)


def hierarchical_match(F: np.ndarray, q: np.ndarray, ratios=(4, 2, 1), tau_sim: float = 0.5, nms: int = 19):
    """CryoSAM-style matching: coarse-to-fine cosine similarity with pruning, then 3D NMS at the finest level."""
    keep, prev_r, evaluated, maps = None, None, [], []
    for r in ratios:
        Fr = downsample_features(F, r)
        S = np.full(Fr.shape[:3], -1.0, np.float32)
        if keep is None:
            cand = np.ones(Fr.shape[:3], bool)                    # coarsest level: evaluate everything
        else:                                                   # finer level: only children of survivors
            f = prev_r // r
            up = keep.repeat(f, 0).repeat(f, 1).repeat(f, 2)
            cand = np.zeros(Fr.shape[:3], bool)
            sl = tuple(slice(0, min(a, b)) for a, b in zip(up.shape, cand.shape))
            cand[sl] = up[sl]
        S[cand] = cosine_to(Fr[cand], q)
        evaluated.append(int(cand.sum())); maps.append(S)
        keep, prev_r = S >= tau_sim, r
    peaks = (S == ndimage.maximum_filter(S, size=nms)) & keep  # 3D non-maximum suppression
    pts, sc = np.argwhere(peaks), S[peaks]
    order = np.argsort(-sc)
    return pts[order], sc[order], maps, evaluated


diam = 2 * (3 * mask_3.sum() / (4 * np.pi)) ** (1 / 3)          # equivalent-sphere diameter of the clicked particle
NMS = int(diam) // 2 * 2 + 1
q_mask = Fw[mask_3].mean(0)                                      # paper: masked average
q_cent = Fw[tuple(np.round(np.argwhere(mask_3).mean(0)).astype(int))]   # released code: feature at the centroid
t0 = time.time()
cand_pts, cand_sc, sim_maps, n_eval = hierarchical_match(Fw, q_cent, nms=NMS)
print(f"clicked particle ≈ {diam:.0f} nm across → NMS window {NMS}³ voxels; matching took {time.time() - t0:.2f} s")
total = np.prod(Fw.shape[:3])
for r, n in zip((4, 2, 1), n_eval):
    print(f"  level 1/{r}: evaluated {n:>8,} feature vectors")
print(f"  brute force at full resolution would evaluate {total:,}; we evaluated {sum(n_eval):,} "
      f"({100 * sum(n_eval) / total:.0f} %)")
print(f"{len(cand_pts)} candidate prompts with similarity ≥ 0.5 after NMS")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, S, r in zip(axes, sim_maps, (4, 2, 1)):
    im = ax.imshow(np.ma.masked_less(S[z_mid // r], -0.99), cmap="viridis", vmin=-0.5, vmax=1)
    ax.set_title(f"similarity at 1/{r} (grey = pruned)"); ax.axis("off"); ax.set_facecolor("0.85")
plt.colorbar(im, ax=axes, fraction=0.02, label="cosine similarity")
plt.show()

🔍 **What just happened?** At the coarsest level every (pooled) voxel was scored; most of the background fell below 0.5
and was never looked at again, so the full-resolution level only evaluated a fraction of the tomogram — the reason the
paper's hierarchical matching cut runtime several-fold. Bright spots in the final map mark places that "look like" the
clicked ribosome.

### 3.4 New prompts → verified detections → evaluation

Each NMS peak becomes a **new point prompt**, and CryoSAM runs SAM on it (step 1 again). We mimic this by running our
self-prompting segmenter (along z, as in the released code) on each candidate and keeping it only if (a) the toy SAM
accepts the click (bright enough) and (b) the resulting mask has a plausible size (between ½× and 2× the clicked
particle's mask). We then score the detections against the answer key with one-to-one matching (Module 11): a
detection is a **true positive** if it lies within 10 nm of an unmatched true ribosome centre — the same 10-voxel rule
used by TomoPicker and the Stable-Diffusion segmentation paper.

$$\text{precision}=\frac{TP}{TP+FP},\qquad \text{recall}=\frac{TP}{TP+FN},\qquad F_1=\frac{2PR}{P+R}$$

In [ ]:
# ── 3.4 Verify candidate prompts with the toy SAM and evaluate ──
HIT_RADIUS = 10.0                                                # nm (= voxels)


def verify_prompts(pts, ref_voxels: int, factor: float = 2.0):
    """Run self-prompting (z only) on each candidate; keep those with an accepted, plausibly sized mask."""
    keep, seg = [], np.zeros(den.shape, bool)
    for p in pts:
        m = self_prompt(p, axis=0)
        ok = ref_voxels / factor <= m.sum() <= ref_voxels * factor
        keep.append(ok)
        if ok:
            seg |= m
    return np.array(keep, bool), seg


def match_flags(pred, true, radius: float):
    """Greedy one-to-one matching (as in match_points) that also returns which predictions / truths were matched."""
    pred, true = np.asarray(pred, float).reshape(-1, 3), np.asarray(true, float)
    tp_pred, found = np.zeros(len(pred), bool), np.zeros(len(true), bool)
    if len(pred) and len(true):
        d = np.linalg.norm(pred[:, None] - true[None], axis=-1)
        for i, j in zip(*np.unravel_index(np.argsort(d, axis=None), d.shape)):
            if d[i, j] > radius:
                break
            if not tp_pred[i] and not found[j]:
                tp_pred[i] = found[j] = True
    return tp_pred, found


t0 = time.time()
ok, seg_all = verify_prompts(cand_pts, int(self_prompt(click, 0).sum()))
det_pts, det_sc = cand_pts[ok], cand_sc[ok]
res = match_points(det_pts, ribo_true, HIT_RADIUS)
tp_flag, found = match_flags(det_pts, ribo_true, HIT_RADIUS)
dice_seg = 2 * (seg_all & gt_ribo_mask).sum() / (seg_all.sum() + gt_ribo_mask.sum())
print(f"verification of {len(cand_pts)} candidates took {time.time() - t0:.1f} s → {len(det_pts)} detections")
print(f"ONE click: precision {res['precision']:.2f}, recall {res['recall']:.2f}, F1 {res['f1']:.2f} "
      f"(TP {res['tp']}, FP {res['fp']}, FN {res['fn']}); voxel Dice of the ribosome segmentation {dice_seg:.2f}")
fp_classes = []
for p in det_pts[~tp_flag]:
    j = np.argmin(np.linalg.norm(centers - p, axis=1))
    fp_classes.append(classes[j] if np.linalg.norm(centers[j] - p) < 10 else "background/membrane")
print("false positives sit on:", {ZOO.get(c, {"name": c})["name"]: fp_classes.count(c) for c in set(fp_classes)})

In [ ]:
# ── 3.4b Map of the detections ──
fig, ax = plt.subplots(figsize=(5.6, 5.4))
ax.imshow(den.mean(0), cmap="gray")
ax.scatter(ribo_true[found, 2], ribo_true[found, 1], s=260, facecolors="none", edgecolors="tab:green", lw=1.5,
           label="true ribosome, found")
ax.scatter(ribo_true[~found, 2], ribo_true[~found, 1], s=260, facecolors="none", edgecolors="tab:red", lw=1.5,
           label="true ribosome, missed (FN)")
ax.scatter(det_pts[tp_flag, 2], det_pts[tp_flag, 1], marker="+", s=90, color="lime", label="detection (TP)")
ax.scatter(det_pts[~tp_flag, 2], det_pts[~tp_flag, 1], marker="x", s=60, color="orange", label="detection (FP)")
ax.plot(click[2], click[1], "*", color="yellow", ms=16, mec="k", label="the one click")
ax.set_title("CryoSAM-lite from one click (z-projection)"); ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
ax.legend(fontsize=7, loc="lower right", framealpha=0.85)
plt.tight_layout(); plt.show()

🔍 **What just happened?** From a single click and **no training at all**, the pipeline found every ribosome in the
cell. Its mistakes are instructive: the false positives sit on dense membrane or background patches and on *other
large particles* (a proteasome can look ribosome-ish to a filter bank) — the features, not the logic, limit precision. That is why the paper's ablation found the choice of
features decisive (DINOv2 > DINO > SAM's own encoder).

### 3.5 More clicks, and the paper-vs-code query

The paper reports that going from one prompt to 5 % of the particles as prompts raises F1. Let us vary (i) the number
of clicks and (ii) the query definition (masked average as in the paper's equation vs centroid sample as in the code),
and draw precision–recall curves by sweeping a final similarity threshold over the verified detections.

In [ ]:
# ── 3.5 Precision–recall curves: 1 vs 3 clicks, masked-average vs centroid query ──
def run_cryosam_lite(click_list, query: str = "centroid"):
    """Full toy pipeline from a list of 3D clicks; returns verified detections and their similarity scores."""
    masks = [segment_3d(c) for c in click_list]
    union = np.any(masks, axis=0)
    if query == "mask":
        q = Fw[union].mean(0)
    else:
        q = np.mean([Fw[tuple(np.round(np.argwhere(m).mean(0)).astype(int))] for m in masks], axis=0)
    d = np.mean([2 * (3 * m.sum() / (4 * np.pi)) ** (1 / 3) for m in masks])
    pts, sc, _, _ = hierarchical_match(Fw, q, nms=int(d) // 2 * 2 + 1)
    ref = int(np.mean([self_prompt(c, 0).sum() for c in click_list]))
    ok, _ = verify_prompts(pts, ref)
    return pts[ok], sc[ok]


def pr_curve(pts, sc, true, radius: float):
    """Precision and recall of the detections above each similarity threshold."""
    P, R = [], []
    for t in np.sort(sc)[::-1]:
        m = match_points(pts[sc >= t], true, radius)
        P.append(m["precision"]); R.append(m["recall"])
    return np.array(P), np.array(R)


t0 = time.time()
clicks3 = [np.round(centers[i]).astype(int) for i in ribo_idx[:3]]
fig, ax = plt.subplots(figsize=(5.2, 3.8))
for clist, query, style in [([click], "centroid", "o-"), ([click], "mask", "s--"), (clicks3, "centroid", "^-")]:
    pts, sc = run_cryosam_lite(clist, query)
    P, R = pr_curve(pts, sc, ribo_true, HIT_RADIUS)
    f1 = 2 * P * R / np.maximum(P + R, 1e-9)
    lab = f"{len(clist)} click{'s' if len(clist) > 1 else ''}, {query} query (best F1 {f1.max():.2f})"
    ax.plot(R, P, style, ms=3, label=lab)
    print(f"{lab}: {len(pts)} verified detections")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02)
ax.set_title("CryoSAM-lite precision–recall (ribosomes)"); ax.legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.show()
print(f"(three pipelines in {time.time() - t0:.1f} s)")

🔍 **What just happened?** Each curve sweeps a final similarity threshold: strict thresholds keep only the most
ribosome-like detections (high precision, low recall). In our toy the **centroid query (code)** usually beats the
**masked average (paper equation)** — averaging over the whole mask mixes the particle's edge and interior voxels, while
the centroid feature describes "the middle of a ribosome", which is what NMS looks for. More clicks should give a more
representative query; here one click already finds every ribosome without a false positive at the best threshold, and
with only eight ribosomes in the slab, one detection more or less moves F1 by several points. Treat these as *toy* findings: with real DINOv2 features (coarse 14-pixel patches) the two
queries behave differently.

✏️ **Your turn.** (a) Click a **proteasome** instead: in cell 3.2 set
`click = np.round(centers[np.where(classes == "3j9i")[0][0]]).astype(int)`; in cell 3.4 set
`ribo_true = centers[classes == "3j9i"]` and `HIT_RADIUS = 7`; re-run 3.2–3.4 and compare precision and recall (the
Dice value and the plot legend still refer to ribosomes). Which class is easiest to find from one click, and why? (b) Change `tau_sim` in `hierarchical_match` from 0.5 to 0.3 or
0.7: how do the number of evaluated vectors and the recall change?

<details><summary>Hint</summary>Ribosomes are the largest and most distinctive particles; proteasomes are elongated and
their size overlaps both other classes, so their features are the least unique. A lower τ_sim keeps more voxels (slower,
more candidates, higher recall but more false prompts to verify); a higher τ_sim prunes harder and can lose true particles
already at the coarse level.</details>

### 📄 Paper connection — CryoSAM

**Paper**: Zhao Y, Bian H, Mu M, Uddin MR, Li Z, Li X, Wang T, Xu M. *CryoSAM: Training-Free CryoET Tomogram
Segmentation with Foundation Models.* MICCAI 2024, LNCS, pp. 124–134. [doi:10.1007/978-3-031-72111-3_12](https://doi.org/10.1007/978-3-031-72111-3_12)
(preprint: *Training-free CryoET Tomogram Segmentation*, [arXiv:2407.06833](https://arxiv.org/abs/2407.06833)).

- **Question it answers:** can every particle of a class be segmented in a tomogram from as little as one click, with
  no training and no annotated dataset?
- **What the paper actually does:** (1) *cross-plane self-prompting* — frozen SAM segments the clicked slice; the most
  confident point of each mask prompts the next slice (the previous mask is also given as a mask prompt); propagation
  stops when adjacent masks have IoU < 0.5; directions are unioned. (2) *hierarchical feature matching* — frozen DINOv2
  features extracted slice by slice along all three axes form a tri-view feature volume; the query is the masked average
  $\mathbf F_Q$; cosine similarity is computed on a 1/16 → 1/8 → 1/4 pyramid with pruning at 0.5; 3D NMS and top-K
  (K = 512) give new prompts that go back to step (1).
- **What our toy keeps:** the whole control flow — self-prompting with IoU stopping, tri-view features, masked-average
  query, pruned coarse-to-fine cosine matching, max-pool NMS, prompts fed back to the segmenter.
- **What our toy leaves out:** the foundation models themselves (replaced by region growing and a whitened filter
  bank), mask prompts, SAM's multi-mask outputs and scores, top-K selection, real data and denoising (Warp).
- **Code vs paper:** the released AITom code propagates SAM only along ±z and samples the query feature at the mask
  *centroid* (then averages over prompts) instead of averaging over all mask voxels; its config uses
  `down_sample_ratios=[16, 8, 4]`, `similarity_threshold=0.5`, `top_k=512`, `iou_threshold=0.5` and an area filter of
  32 × 32 pixels.
- **Result reported:** on EMPIAR-10499 (*Mycoplasma pneumoniae* cells, 70S ribosomes) the authors report F1 = 54.2 from a
  single prompt (< 1 % of annotations) — above crYOLO trained on all labels (52.0) and Huang *et al.* with 10 % labels
  (53.8) — and about 65 F1 with 5 % prompts. In their feature ablation DINOv2 (57.1) > DINO (54.5) > SAM encoder (38.1),
  and hierarchical matching cut runtime from about 60 to 10–15 min. (The paper's 10 %-label F1 appears as 65.5 in one
  table and 68.5 in others — a small inconsistency worth noticing.)
- **Code:** [`aitom/segmentation/foundation_model/cryosam`](https://github.com/xulabs/aitom/tree/master/aitom/segmentation/foundation_model/cryosam)

---
## 4 · Segmentation from attention eigenvectors (Stable Diffusion, zero labels)

### 4.1 The idea: attention is an affinity matrix

Inside a transformer, every image patch $i$ sends a **query** vector and every patch $j$ a **key** vector; the
**self-attention** weight is

$$A_{ij}=\frac{\exp\!\big(q_i\cdot k_j/\sqrt d\big)}{\sum_{j'}\exp\!\big(q_i\cdot k_{j'}/\sqrt d\big)},\qquad\text{i.e.}\quad A=\mathrm{softmax}\!\Big(\frac{QK^\top}{\sqrt d}\Big)\ \text{row by row.}$$

*In words:* patch $i$ distributes one unit of "attention" over all patches, giving more to patches whose key matches its
query; $d$ is the vector length and $\sqrt d$ keeps the numbers in a sensible range. Each row of $A$ sums to 1, so $A$ is
like a **random walk**: from patch $i$, jump to patch $j$ with probability $A_{ij}$. In Stable Diffusion's U-Net, patches
of the same object attend strongly to each other — so a random walker tends to stay *inside* an object.

🧮 **Eigenvectors in one paragraph.** A vector $z$ is an *eigenvector* of $A$ with *eigenvalue* $\lambda$ if $Az=\lambda z$:
applying the matrix only rescales it. For a random-walk matrix the largest eigenvalue is exactly 1 (with $z$ = a constant
vector: "everyone stays in the image"). If the image consists of groups that rarely exchange walkers, there are several
eigenvalues close to 1 whose eigenvectors are **almost constant within each group** — so plotting them as images
*paints the groups*. This is **spectral clustering**.

The Stable-Diffusion paper (Uddin *et al.*, CVPR 2026) does not call `numpy.linalg.eig`. Its code learns $K$ maps
$z_1,\dots,z_K$ (produced by a small CNN, made non-negative and summing to one per pixel with a softmax, then each scaled
to unit length) by minimising

$$\mathcal L=w_e\cdot\frac1K\sum_k\big|z_k^\top A\,z_k-1\big|\;+\;w_o\cdot\operatorname{mean}_{i\ne j}\big(z_i^\top z_j\big)^2,\qquad w_e=w_o=2.$$

*In words:* make each map behave like an eigenvector with eigenvalue ≈ 1 (first term) while keeping the maps
different from one another (orthogonal, second term). Let us check that both routes find the same structure.

Our toy: a 64 × 64-nm slab with a plasma-membrane-like arc, a vesicle and ~22 small particles (2–3 nm radius) in noise
(SNR 0.5). Our "attention" uses the filter-bank features of 2 × 2-pixel patches (32 × 32 = 1,024 tokens) as both
queries and keys, $A=\mathrm{softmax}(FF^\top/\sqrt d)$; we clip extreme feature values at ±2 so that no patch
attends only to itself.

In [ ]:
# ── 4.1 A membrane-rich slab with known answers ──
def make_membrane_slice(rng: np.random.Generator, n: int = 64, n_particles: int = 22, snr: float = 0.5):
    """2D slab: a curved membrane, a vesicle and small round particles (radius 2.2–3.2 px), blurred, plus noise.

    Returns (noisy image, membrane mask, particle mask, particle centres)."""
    yy, xx = np.mgrid[0:n, 0:n].astype(float)
    mem = np.abs(np.sqrt((yy + 70) ** 2 + (xx - 30) ** 2) - 95) < 1.6        # gently curved 'plasma membrane'
    mem |= np.abs(np.sqrt((yy - 40) ** 2 + (xx - 44) ** 2) - 11) < 1.5       # a vesicle
    occupied = ndimage.binary_dilation(mem, iterations=4) | (np.sqrt((yy - 40) ** 2 + (xx - 44) ** 2) < 12)
    part, cents, tries = np.zeros((n, n), bool), [], 0
    while len(cents) < n_particles and tries < 5000:
        tries += 1
        c, r = rng.uniform(4, n - 4, 2), rng.uniform(2.2, 3.2)
        disk = (yy - c[0]) ** 2 + (xx - c[1]) ** 2 <= r * r
        if (disk & ndimage.binary_dilation(occupied | part, iterations=2)).any():
            continue
        part |= disk; cents.append(c)
    img = ndimage.gaussian_filter((mem | part).astype(float), 1.0)
    noisy = img + rng.normal(0, np.sqrt(img.var() / snr), img.shape)
    return noisy.astype(np.float32), mem, part, np.array(cents)


def features_2d(im: np.ndarray, scales=(1, 2, 4)) -> np.ndarray:
    """2D filter bank (blur, LoG, gradient magnitude, coherence) at 3 scales → array[H, W, 12]."""
    out = []
    for s in scales:
        out += [ndimage.gaussian_filter(im, s), ndimage.gaussian_laplace(im, s) * s * s,
                ndimage.gaussian_gradient_magnitude(im, s) * s]
        d1, d2 = ndimage.gaussian_filter(im, s, order=(1, 0)), ndimage.gaussian_filter(im, s, order=(0, 1))
        j11, j22, j12 = (ndimage.gaussian_filter(a, 2 * s) for a in (d1 * d1, d2 * d2, d1 * d2))
        out.append(np.sqrt((j11 - j22) ** 2 + 4 * j12 ** 2) / (j11 + j22 + 1e-6))
    return np.stack(out, -1)


def attention_matrix(img: np.ndarray, patch: int = 2, clip: float = 2.0):
    """Row-stochastic 'self-attention' A = softmax(F Fᵀ/√d) between patch tokens, plus the pieces needed for eig."""
    n = img.shape[0] // patch
    F = features_2d(img).reshape(n, patch, n, patch, -1).mean((1, 3)).reshape(n * n, -1)
    F = np.clip((F - F.mean(0)) / (F.std(0) + 1e-6), -clip, clip)
    logits = F @ F.T / np.sqrt(F.shape[1])
    W = np.exp(logits - logits.max())          # symmetric; softmax rows = W / row sums (the max cancels)
    deg = W.sum(1)
    return W / deg[:, None], W, deg, n


img2d, mem_gt, part_gt, part_centers = make_membrane_slice(np.random.default_rng(0))
A, W_aff, deg, n_tok = attention_matrix(img2d)
print(f"{len(part_centers)} particles; attention matrix {A.shape}, rows sum to 1: {np.allclose(A.sum(1), 1)}; "
      f"largest self-attention A_ii = {A.diagonal().max():.2f}")

In [ ]:
# ── 4.2 Route 1: eigenvectors by linear algebra ──
K_EIG = 6
t0 = time.time()
# A = D⁻¹W has the same eigenvalues as the symmetric S = D^(-1/2) W D^(-1/2); its eigenvectors are D^(-1/2)·u.
S_sym = W_aff / np.sqrt(deg[:, None] * deg[None, :])
vals, U = eigsh(S_sym, k=K_EIG, which="LA")
order = np.argsort(-vals); vals, U = vals[order], U[:, order]
V_eig = U / np.sqrt(deg)[:, None]
V_eig /= np.linalg.norm(V_eig, axis=0)
print(f"top-{K_EIG} eigenvalues of A in {time.time() - t0:.2f} s:", np.round(vals, 3))
print("check A·v = λ·v:", np.allclose(A @ V_eig, V_eig * vals, atol=1e-6))

In [ ]:
# ── 4.3 Route 2: the paper-style loss |zᵀAz − 1| + orthogonality, optimised with PyTorch ──
class EigenMapNet(nn.Module):
    """Tiny CNN with a learnable input tensor that outputs K maps (a smaller version of the lab's ToyCNN)."""

    def __init__(self, h: int, w: int, k: int, hidden: int = 32):
        super().__init__()
        self.inp = nn.Parameter(torch.randn(1, hidden, h, w))
        self.c1 = nn.Conv2d(hidden, hidden, 3, padding=1)
        self.bn = nn.BatchNorm2d(hidden)
        self.c2 = nn.Conv2d(hidden, k, 3, padding=1)

    def forward(self):
        return self.c2(torch.relu(self.bn(self.c1(self.inp))))


torch.manual_seed(SEED)
A_t = torch.tensor(A, dtype=torch.float32, device=DEVICE)
net = EigenMapNet(n_tok, n_tok, K_EIG).to(DEVICE)
opt = torch.optim.Adam(net.parameters(), lr=5e-3)
off_diag = ~torch.eye(K_EIG, dtype=torch.bool, device=DEVICE)
t0, history = time.time(), []
for step in range(300):
    Z = Fnn.normalize(net().softmax(1).reshape(K_EIG, -1), dim=1)   # soft partition, unit length per map
    rayleigh = ((Z @ A_t) * Z).sum(1)                                 # z_kᵀ A z_k for each map
    loss = 2 * (rayleigh - 1).abs().mean() + 2 * ((Z @ Z.T)[off_diag] ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    history.append(loss.item())
Z_learned = Z.detach().cpu().numpy().T                                # (tokens, K)
print(f"300 Adam steps in {time.time() - t0:.1f} s; final loss {history[-1]:.3f}; "
      f"zᵀAz per map: {np.round(rayleigh.detach().cpu().numpy(), 2)}")
# Compare the two K-dimensional subspaces through their principal angles
Q1, _ = np.linalg.qr(Z_learned)
Q2, _ = np.linalg.qr(V_eig)
cosines = np.linalg.svd(Q1.T @ Q2, compute_uv=False)
print("cosines of the principal angles between the two subspaces:", np.round(cosines, 2))
assert cosines[:3].min() > 0.8, "the learned maps should span the leading eigenvectors"

fig, axes = plt.subplots(2, K_EIG + 1, figsize=(13, 4.2))
axes[0, 0].imshow(img2d); axes[0, 0].set_title("noisy slab", fontsize=8)
axes[1, 0].imshow(mem_gt * 1 + part_gt * 2, cmap="viridis"); axes[1, 0].set_title("truth (membrane/particle)", fontsize=8)
for k in range(K_EIG):
    v_img = V_eig[:, k].reshape(n_tok, n_tok)
    m_ = v_img.mean()
    lim = dict(vmin=m_ - abs(m_), vmax=m_ + abs(m_) + 1e-12) if k == 0 else {}   # eigenvector 1 is constant (sign arbitrary): don't stretch float noise
    axes[0, k + 1].imshow(v_img, cmap="viridis", **lim)
    axes[0, k + 1].set_title(f"eigenvector {k + 1}, λ = {vals[k]:.2f}", fontsize=8)
    axes[1, k + 1].imshow(Z_learned[:, k].reshape(n_tok, n_tok), cmap="viridis")
    axes[1, k + 1].set_title(f"learned map {k + 1}", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Top: eigenvectors of A (linear algebra).  Bottom: maps learned with the paper-style loss.", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The first eigenvector (λ = 1) is constant — the trivial "stay in the image" solution. The
next ones are *pictures*: one separates background from everything else; others pick out the membranes or the small
particles. The learned maps look different one by one (any rotation of a set of eigenvectors spans the same space, and
the softmax makes the maps non-negative), but the **principal-angle cosines** show that the leading directions of the
two subspaces coincide (cosine ≈ 1). The last, weakest directions agree less — eigenvectors with small eigenvalues are
poorly determined by both methods. Why learn instead of `eig`? Real attention matrices come from many layers and
resolutions (the code sums the loss over all of them) and are large; gradient descent handles both gracefully.

### 4.4 From eigenvectors to masks: adaptive threshold, instances and circularity

Following the released pipeline in a simplified form, we (1) **pick** the eigenvector whose histogram is most clearly
two-peaked (a simplification of the paper's heuristic that scores and merges several eigenvectors), flipping its sign
so that the minority — the structures — is bright; (2) blur it and apply an **adaptive Gaussian threshold** (each pixel
is compared with a Gaussian-weighted mean of its 15 × 15 neighbourhood); (3) split touching objects into **instances**
(the code uses the pretrained Cellpose model; we use a classical watershed on the distance transform as a stand-in); and
(4) call an instance a **particle** if its circularity

$$\text{circularity}=\frac{4\pi\,\text{area}}{\text{perimeter}^2}\ \ge\ 0.95,$$

(1 for a perfect disc; small for long thin shapes such as membranes) and if it is not glued to a membrane; all
remaining foreground is **membrane**. *(Small digital discs can score slightly above 1 because pixelated perimeters are
underestimated.)*

In [ ]:
# ── 4.4 Eigenvector → masks → membranes vs particles ──
def eigenvector_maps(img: np.ndarray, k: int = K_EIG, patch: int = 2):
    """Top-k right eigenvectors of the attention matrix of `img`, upsampled to pixel resolution."""
    A_, W_, d_, n_ = attention_matrix(img, patch)
    lam, U_ = eigsh(W_ / np.sqrt(d_[:, None] * d_[None, :]), k=k, which="LA")
    o = np.argsort(-lam)
    V_ = U_[:, o] / np.sqrt(d_)[:, None]
    return lam[o], np.stack([ndimage.zoom(V_[:, i].reshape(n_, n_), patch, order=1) for i in range(k)])


def pick_eigenvector(maps: np.ndarray):
    """Choose the most two-peaked non-trivial eigenvector (Otsu between-class variance), structures made bright."""
    best = None
    for i, m in enumerate(maps[1:], 1):
        m = (m - m.min()) / (m.max() - m.min() + 1e-12)
        t = threshold_otsu(m)
        if (m > t).mean() > 0.5:                     # conditional inversion: structures are the minority
            m, t = 1 - m, 1 - t
        fg = m > t
        score = fg.mean() * (1 - fg.mean()) * (m[fg].mean() - m[~fg].mean()) ** 2 / m.var()
        if best is None or score > best[0]:
            best = (score, i, m)
    return best[1], best[2]


def split_instances(fg: np.ndarray, min_distance: int = 3) -> np.ndarray:
    """Watershed on the distance transform: a classical stand-in for Cellpose instance splitting."""
    dist = ndimage.distance_transform_edt(fg)
    peaks = peak_local_max(dist, min_distance=min_distance, labels=sk_label(fg))
    markers = np.zeros(fg.shape, int)
    markers[tuple(peaks.T)] = np.arange(1, len(peaks) + 1)
    return watershed(-dist, markers, mask=fg)


def membrane_particle_split(fg: np.ndarray, lab: np.ndarray, circ_thr: float = 0.95, min_area: int = 6):
    """Round instances away from non-round structures → particles; the rest of the foreground → membrane."""
    other = np.zeros(fg.shape, bool); round_regions = []
    for r in regionprops(lab):
        circ = 4 * np.pi * r.area / max(r.perimeter, 1) ** 2
        if circ >= circ_thr and r.area >= min_area:
            round_regions.append(r)
        else:
            other[lab == r.label] = True
    dist_other = ndimage.distance_transform_edt(~other)
    centroids, pmask = [], np.zeros(fg.shape, bool)
    for r in round_regions:
        if dist_other[tuple(np.round(r.centroid).astype(int))] >= 1.0 + np.sqrt(r.area / np.pi):
            centroids.append(r.centroid); pmask |= lab == r.label
    return fg & ~pmask, pmask, np.array(centroids).reshape(-1, 2)


def unsupervised_segment(img: np.ndarray):
    """Zero-label pipeline: attention eigenvectors → adaptive threshold → instances → circularity split."""
    lam, maps = eigenvector_maps(img)
    k, m = pick_eigenvector(maps)
    sm = ndimage.gaussian_filter(m, 1.5)
    fg = sm > threshold_local(sm, 15, method="gaussian", offset=0.0)
    memb, pmask, cents = membrane_particle_split(fg, split_instances(fg))
    return dict(k=k, map=m, fg=fg, membrane=memb, particles=pmask, centroids=cents)


def dice(a: np.ndarray, b: np.ndarray) -> float:
    """Dice coefficient 2|A∩B|/(|A|+|B|) of two boolean masks."""
    return float(2 * (a & b).sum() / (a.sum() + b.sum() + 1e-12))


t0 = time.time()
seg = unsupervised_segment(img2d)
mem_ref = ndimage.binary_dilation(mem_gt)                       # a 1-pixel tolerance band around thin membranes
r2 = match_points(seg["centroids"], part_centers, 3.0)
print(f"zero-label segmentation in {time.time() - t0:.2f} s using eigenvector {seg['k'] + 1}")
print(f"membrane Dice {dice(seg['membrane'], mem_ref):.2f} | particles: precision {r2['precision']:.2f}, "
      f"recall {r2['recall']:.2f}, F1 {r2['f1']:.2f} (match radius 3 px)")

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
axes[0].imshow(seg["map"], cmap="viridis"); axes[0].set_title(f"chosen eigenvector ({seg['k'] + 1})")
axes[1].imshow(seg["fg"]); axes[1].set_title("adaptive threshold")
overlay = np.zeros(img2d.shape + (3,)); overlay[seg["membrane"]] = (0.2, 0.6, 1); overlay[seg["particles"]] = (1, 0.5, 0)
axes[2].imshow(overlay); axes[2].set_title("membrane (blue) / particle (orange)")
axes[3].imshow(img2d); axes[3].contour(mem_gt, [0.5], colors="cyan", linewidths=0.8)
axes[3].scatter(part_centers[:, 1], part_centers[:, 0], s=12, c="red", label="true particles")
axes[3].scatter(seg["centroids"][:, 1], seg["centroids"][:, 0], s=40, facecolors="none", edgecolors="yellow", label="found")
axes[3].set_title("truth vs found"); axes[3].legend(fontsize=6, loc="lower left")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With **no labels at all**, the attention eigenvector outlined both scales of structure —
long membranes and nanometre-sized particles — and a single geometric rule (circularity) told them apart. That is the
sense of "multi-scale" in the paper's title. Errors come from particles touching the membrane (deliberately discarded,
as in the code) and from membrane fragments that happen to look round.

### 4.5 Pseudo-labels → a fast supervised model

The eigen pipeline is slow on real data (a diffusion model forward pass and an optimisation per slice), so the paper
runs it on a few representative slabs only and uses its output as **pseudo-labels** to train fast supervised models
(a membrane U-Net and the DeepETPicker particle picker), which then label the remaining tomograms. Our miniature: train a
per-pixel logistic-regression classifier (background / membrane / particle) on the filter-bank features of **two**
pseudo-labelled slabs, then apply it to six new slabs it has never seen.

In [ ]:
# ── 4.5 Train on pseudo-labels, test on new slabs ──
X_train, y_train = [], []
for s in (0, 1):
    im, *_ = make_membrane_slice(np.random.default_rng(s))
    sg = unsupervised_segment(im)
    y = np.zeros(im.shape, int); y[sg["membrane"]] = 1; y[sg["particles"]] = 2      # pseudo-labels, no human input
    X_train.append(features_2d(im).reshape(-1, 12)); y_train.append(y.ravel())
X_train, y_train = np.concatenate(X_train), np.concatenate(y_train)
mu, sd = X_train.mean(0), X_train.std(0)
clf = LogisticRegression(max_iter=500).fit((X_train - mu) / sd, y_train)

rows = []
for s in range(2, 8):
    im, mem_s, _, cent_s = make_membrane_slice(np.random.default_rng(s))
    t0 = time.time(); sg = unsupervised_segment(im); t_eig = time.time() - t0
    t0 = time.time()
    pred = clf.predict((features_2d(im).reshape(-1, 12) - mu) / sd).reshape(im.shape)
    inst = split_instances(ndimage.binary_opening(pred == 2))
    cents = np.array([r.centroid for r in regionprops(inst) if r.area >= 6]).reshape(-1, 2)
    t_sup = time.time() - t0
    band = ndimage.binary_dilation(mem_s)
    rows.append((dice(sg["membrane"], band), match_points(sg["centroids"], cent_s, 3)["f1"], t_eig,
                 dice(pred == 1, band), match_points(cents, cent_s, 3)["f1"], t_sup))
rows = np.array(rows)
print("slab | eigen pipeline: membrane Dice, particle F1, time | supervised on pseudo-labels: Dice, F1, time")
for s, r in zip(range(2, 8), rows):
    print(f"  {s}  |  {r[0]:.2f}   {r[1]:.2f}   {r[2]:.2f} s  |  {r[3]:.2f}   {r[4]:.2f}   {r[5]:.3f} s")
print(f"mean |  {rows[:, 0].mean():.2f}   {rows[:, 1].mean():.2f}            |  {rows[:, 3].mean():.2f}   {rows[:, 4].mean():.2f}")

🔍 **What just happened?** The cheap classifier, trained only on machine-made labels from two slabs, labels new slabs
several times faster — and here it even *beats* the pipeline that taught it, because training averages away the
teacher's slab-specific mistakes. The paper's ablation shows the same direction on real *S. pombe* tomograms (removing
the pseudo-label training lowered membrane Dice and macromolecule F1). ⚠️ Pseudo-labels can also teach *systematic*
errors: whatever the eigen pipeline always gets wrong (e.g. particles touching membranes), the student learns too.

### 📄 Paper connection — unsupervised multi-scale segmentation with Stable Diffusion

**Paper**: Uddin MR, Tabib HMS, Nguyen T-H, Gandhi K, Xu M. *Unsupervised Multi-Scale Segmentation of 3D Subcellular World
with Stable Diffusion Foundation Model.* CVPR 2026, pp. 22744–22752 ([CVF page](https://openaccess.thecvf.com/content/CVPR2026/html/Uddin_Unsupervised_Multi-Scale_Segmentation_of_3D_Subcellular_World_with_Stable_Diffusion_CVPR_2026_paper.html));
an earlier version appeared on bioRxiv (June 2025). The CVPR poster page marks it a *Highlight* (the AITom README says
"Spotlight").

> ⚠️ We could not access the main paper text while writing; the description below is limited to the abstract, the
> supplementary material and the released code. Read the paper before relying on details.

- **Question it answers:** can membranes *and* macromolecules be segmented in cellular tomograms with **no labels**, and
  can the result replace human annotation for training fast supervised tools?
- **What it does (code + supplement):** slices are contrast-stretched, smoothed and fed (as RGB patches) to Stable
  Diffusion v1.4; the latent is noised at a random, small diffusion timestep and the U-Net runs with an empty text prompt; **self-attention
  maps** from all layers are collected; a small CNN learns $K=12$ soft maps with the loss
  $2\,\mathrm{mean}_k|z_k^\top A z_k-1| + 2\,\mathrm{mean}_{i\neq j}(z_i^\top z_j)^2$ (approximate leading eigenvectors, i.e.
  spectral clustering); a heuristic selects and merges eigenvectors (Otsu contour counts, patch diversity, SSIM/PSNR
  rules) and smooths across z; an adaptive Gaussian threshold (block 15, C = 3), pretrained **Cellpose** instances and a
  **circularity ≥ 0.95** rule split macromolecules from membranes; the results become pseudo-labels for a membrane U-Net
  and DeepETPicker.
- **What our toy keeps:** row-stochastic attention, the eigen loss with orthogonality, eigenvector → adaptive threshold →
  instances → circularity split, and pseudo-label training of a faster model.
- **What our toy leaves out:** Stable Diffusion itself (our "attention" is built from a filter bank), multi-layer
  attention, the full eigenvector-merging heuristic and z-smoothing, Cellpose (replaced by watershed), and 3D tomograms.
- **Result reported:** the abstract reports performance close to expert human annotation. The supplement's ablation on
  two *S. pombe* tomograms (EMPIAR-10988 = CZ cryoET Data Portal DS-10001) gives membrane Dice / macromolecule F1 of
  0.487 / 0.394 and 0.564 / 0.475 for the full method, versus 0.441 / 0.171 and 0.471 / 0.391 without pseudo-label
  training, and 0.431 / 0.145 and 0.454 / 0.318 without feature aggregation.
- **Code:** [`aitom/segmentation/unsup_multiscale`](https://github.com/xulabs/aitom/tree/master/aitom/segmentation/unsup_multiscale)

---
## 5 · Topology: is the *shape* right?

### 5.1 Betti numbers

Biology cares about shape in a way voxel counts do not: a mitochondrion with a broken outer membrane is a different
biological claim from an intact one; two touching vesicles are not one vesicle. **Topology** describes shape by counting
features that do not change when you stretch or bend an object (without tearing or gluing). The counts are the
**Betti numbers**:

| | in 2D images | in 3D volumes | example |
|---|---|---|---|
| $\beta_0$ | connected pieces | connected pieces | two separate vesicles → $\beta_0=2$ |
| $\beta_1$ | holes (enclosed background) | tunnels / handles | a ring in 2D → $\beta_1=1$; a donut in 3D → $\beta_1=1$ |
| $\beta_2$ | — | enclosed cavities | a closed vesicle membrane (hollow ball) → $\beta_2=1$ |

💡 A 2D slice through a closed vesicle shows a **ring** ($\beta_1=1$); in 3D the same object is a **shell enclosing a
cavity** ($\beta_2=1$). In both cases the "enclosed background" is what a broken membrane loses.

**Computing them on a pixel grid.** $\beta_0$ = number of connected foreground components. Holes are background
components that do **not** touch the image border. One subtlety: neighbourhoods must be *dual* — if foreground pixels
connect through corners (8-connectivity in 2D, 26 in 3D), background pixels may only connect through edges (4, resp. 6),
otherwise a diagonal line would both enclose and not enclose a region.

In [ ]:
# ── 5.1 Betti numbers in 2D (and 3D) ──
CONN8 = np.ones((3, 3), int)                                   # foreground: 8-connectivity
CONN4 = ndimage.generate_binary_structure(2, 1)                # background: 4-connectivity


def betti_2d(mask: np.ndarray):
    """(β0, β1) of a 2D binary mask: foreground components (8-conn) and enclosed background components (4-conn)."""
    b0 = ndimage.label(mask, structure=CONN8)[1]
    lab, nb = ndimage.label(~mask, structure=CONN4)
    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))) - {0}
    return int(b0), int(nb - len(border))


def betti_3d(mask: np.ndarray):
    """(β0, β2) of a 3D binary mask: components (26-conn) and enclosed cavities (6-conn background)."""
    b0 = ndimage.label(mask, structure=np.ones((3, 3, 3), int))[1]
    lab, nb = ndimage.label(~mask, structure=ndimage.generate_binary_structure(3, 1))
    faces = np.concatenate([lab[0].ravel(), lab[-1].ravel(), lab[:, 0].ravel(), lab[:, -1].ravel(),
                            lab[:, :, 0].ravel(), lab[:, :, -1].ravel()])
    return int(b0), int(nb - len(set(faces) - {0}))


n2 = 64
yy, xx = np.mgrid[0:n2, 0:n2]
rr, ang = np.sqrt((yy - 32) ** 2 + (xx - 32) ** 2), np.arctan2(yy - 32, xx - 32)
ring = np.abs(rr - 18) < 2.5
examples = {
    "vesicle (ring)": ring,
    "broken membrane": ring & (np.abs(ang - 0.6) > 0.15),
    "two vesicles": (np.abs(np.sqrt((yy - 32) ** 2 + (xx - 18) ** 2) - 11) < 2) | (np.abs(np.sqrt((yy - 32) ** 2 + (xx - 46) ** 2) - 11) < 2),
    "two vesicles fused": (np.abs(np.sqrt((yy - 32) ** 2 + (xx - 20) ** 2) - 12) < 2) | (np.abs(np.sqrt((yy - 32) ** 2 + (xx - 44) ** 2) - 12) < 2),
    "ring + 2 specks": ring | ((yy - 8) ** 2 + (xx - 8) ** 2 < 5) | ((yy - 55) ** 2 + (xx - 52) ** 2 < 5),
}
fig, axes = plt.subplots(1, len(examples), figsize=(12, 2.7))
for ax, (name, m) in zip(axes, examples.items()):
    b0, b1 = betti_2d(m)
    ax.imshow(m); ax.set_title(f"{name}\nβ0 = {b0}, β1 = {b1}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()
shell = np.abs(np.sqrt(sum((g - 16) ** 2 for g in np.mgrid[0:33, 0:33, 0:33])) - 10) < 1.5
print("3D: a closed vesicle shell has (β0, β2) =", betti_3d(shell),
      "| the same shell with its top cap cut off:", betti_3d(shell & (np.mgrid[0:33, 0:33, 0:33][0] < 24)))

### 5.2 High Dice, wrong topology

Now a realistic failure: a network's probability map for the vesicle has a small **gap** (a weak stretch of membrane)
and two confident **specks** of noise. Watch the voxel score and the topology disagree.

In [ ]:
# ── 5.2 A prediction that Dice loves and topology hates ──
noise_rng = np.random.default_rng(3)
gt_ring = ring
P = ndimage.gaussian_filter(gt_ring.astype(float), 1.2) * 0.9 + 0.05            # a good, slightly blurry prediction
P *= 1 - 0.75 * np.exp(-((ang - 0.6) ** 2) / (2 * 0.06 ** 2))                    # … with a weak stretch (gap)
for c in [(10, 12), (52, 50)]:                                                  # … and two confident specks
    P += 0.75 * np.exp(-((yy - c[0]) ** 2 + (xx - c[1]) ** 2) / (2 * 1.3 ** 2))
P = np.clip(P + ndimage.gaussian_filter(noise_rng.normal(0, 0.08, P.shape), 1), 0, 1)
pred = P >= 0.5
print(f"Dice = {dice(pred, gt_ring):.3f}   |   Betti (β0, β1): truth {betti_2d(gt_ring)}, prediction {betti_2d(pred)}")

🔍 **What just happened?** A Dice of ≈ 0.98 would be celebrated in most segmentation papers, yet the prediction claims
**three objects and no enclosed lumen** — biologically, an open, broken vesicle plus two extra particles. A topology
error is often only a handful of pixels, which is exactly why pixel-counting scores miss it.

### 5.3 Persistent homology: which features are real?

A probability map does not have *one* topology — it has one for every threshold $t$. Imagine lowering $t$ from 1 to 0
like a falling water level revealing islands (the set $\{P\ge t\}$ grows):

- an island (component) is **born** when $t$ reaches its peak value, and **dies** when it merges into an older island
  (the *elder rule*: the island with the higher peak survives);
- a hole is **born** when a loop closes and **dies** when it fills in.

Each feature is a pair (birth, death); its **persistence** is $|\,\text{birth}-\text{death}\,|$. Plotting all pairs gives a
**persistence diagram**: long-lived features (far from the diagonal) are robust structure, short-lived ones (near the
diagonal) are noise. We compute it with a classic **union–find** algorithm: visit pixels from brightest to darkest and
merge each one with its already-visited neighbours. Holes are handled by a neat duality: *holes of the foreground are
islands of the background*, so we run the same algorithm on the background, darkest first, with the image border acting
as one big "outside" island.

In [ ]:
# ── 5.3 Persistence by union–find (0-dimensional, elder rule) ──
def persistence_pairs(values: np.ndarray, connectivity: int = 8, brightest_first: bool = True,
                      outside_island: bool = False):
    """Birth/death pairs of connected components while sweeping a threshold through `values`.

    brightest_first=True  → superlevel sets {v ≥ t} (components of the foreground, β0).
    brightest_first=False → sublevel sets {v < t}; with outside_island=True the border is one old island, so the
    pairs are the *holes* of the foreground (β1) by duality. Returns a list of (birth, death); the oldest island never
    dies and is omitted."""
    H, Wd = values.shape
    v = values.ravel()
    order = np.argsort(-v) if brightest_first else np.argsort(v)
    parent = np.full(v.size + 1, -1)
    birth = np.zeros(v.size + 1)
    OUT = v.size                                               # index of the virtual 'outside' island
    if outside_island:
        parent[OUT], birth[OUT] = OUT, (np.inf if brightest_first else -np.inf)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    nbrs = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if connectivity == 8 else [])
    older = (lambda a: -birth[a]) if brightest_first else (lambda a: birth[a])
    pairs = []
    for idx in order:
        y, x = divmod(int(idx), Wd)
        parent[idx], birth[idx] = idx, v[idx]
        roots = {idx}
        for dy, dx in nbrs:
            yy_, xx_ = y + dy, x + dx
            if 0 <= yy_ < H and 0 <= xx_ < Wd:
                j = yy_ * Wd + xx_
                if parent[j] >= 0:
                    roots.add(find(j))
            elif outside_island:
                roots.add(find(OUT))
        if len(roots) > 1:
            roots = sorted(roots, key=older)
            for r in roots[1:]:                                 # younger islands die now
                if birth[r] != v[idx]:
                    pairs.append((birth[r], v[idx]))
                parent[r] = roots[0]
    return pairs


t0 = time.time()
pairs0 = persistence_pairs(P)                                   # components (superlevel)
pairs1 = [(d, b) for b, d in persistence_pairs(P, connectivity=4, brightest_first=False, outside_island=True)]
print(f"union–find on {P.size} pixels: {time.time() - t0:.2f} s")
top0 = sorted(pairs0, key=lambda p: -(p[0] - p[1]))[:3]
top1 = sorted(pairs1, key=lambda p: -(p[0] - p[1]))[:2]
print("most persistent extra components (birth → death):", [(round(b, 2), round(d, 2)) for b, d in top0])
print("most persistent hole (born when t drops to → dies at):", [(round(b, 2), round(d, 2)) for b, d in top1])

ts = np.linspace(0.95, 0.05, 37)
curves = np.array([betti_2d(P >= t) for t in ts])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].step(ts, curves[:, 0], where="mid", label="β0 (components)")
axes[0].step(ts, curves[:, 1], where="mid", label="β1 (holes)")
axes[0].axvline(0.5, color="gray", ls=":", label="our threshold 0.5")
axes[0].invert_xaxis(); axes[0].set_xlabel("threshold t (lowered from left to right)"); axes[0].set_ylabel("count")
axes[0].set_title("Betti curves of the probability map"); axes[0].legend(fontsize=7)
b0s = np.array(pairs0); b1s = np.array(pairs1)
axes[1].scatter(b0s[:, 0], b0s[:, 1], s=12, label="components (β0)")
axes[1].scatter(b1s[:, 0], b1s[:, 1], s=25, marker="s", label="holes (β1)")
axes[1].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[1].axvline(0.5, color="gray", ls=":"); axes[1].axhline(0.5, color="gray", ls=":")
axes[1].set_xlabel("birth (threshold where it appears)"); axes[1].set_ylabel("death (threshold where it disappears)")
axes[1].set_title("persistence diagram"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The Betti curves tell the story of the threshold sweep: at high thresholds the ring appears in
pieces; the two specks are separate components until $t$ is very low; and the ring's hole only appears once $t$ drops
below the probability inside the gap (≈ 0.3) — i.e. **the hole is real but slightly too weak** for our 0.5 threshold.
In the diagram, most points hug the diagonal (tiny bumps of noise); a few are far from it: the specks (born ≈ 0.8, they
die only when they merge with the background level) and the hole (born ≈ 0.3). A topology-aware method needs a
**target** (here: one component, one hole) to decide which points to keep.

### 5.4 A TopoFuse-style repair: projection, not penalty, with a certificate

Topology *losses* during training make errors rarer but cannot guarantee they are gone. TopoFuse (Salla *et al.*,
ECCV 2026) adds a **projection** at the end, ProjT: find the smallest change to the prediction whose persistence
diagram matches the target,

$$Z'=\arg\min_{Z^*}\|Z^*-Z\|_2^2\quad\text{s.t.}\quad d_B\big(\mathrm{Dgm}(Z^*),\,\Pi^\star\big)\le\epsilon,$$

*in words:* change the probability map as little as possible (squared differences) such that its diagram is within
$\epsilon$ of the target diagram $\Pi^\star$ ($d_B$, the *bottleneck distance*, is the largest shift needed to match the
points of two diagrams). TopoFuse approximates this with **sparse edits only at the critical voxels** of unmatched
persistence pairs, iterates at most 5 times, and returns a **certificate**: iterations used, fraction of voxels edited,
and whether it converged.

Our 2D toy repair, with target $(\beta_0,\beta_1)=(1,1)$:
- **extra components** → lower the values of the least massive surplus components just below the threshold (only their
  own pixels are touched);
- **missing hole** → find the cheapest path that closes the gap — between the two ends of the skeleton of the broken
  ring, where crossing a pixel costs how much it would have to be raised — and raise only the pixels on that path.

In [ ]:
# ── 5.4 Sparse topology repair with a certificate ──
def skeleton_endpoints(mask: np.ndarray) -> np.ndarray:
    """End points of the morphological skeleton (pixels with exactly one skeleton neighbour)."""
    sk = skeletonize(mask)
    nb = ndimage.convolve(sk.astype(int), np.ones((3, 3), int), mode="constant") - 1
    return np.argwhere(sk & (nb == 1))


def topo_repair(P: np.ndarray, target=(1, 1), thr: float = 0.5, margin: float = 0.02, max_iter: int = 5):
    """Greedy sparse edits until the Betti numbers of {P ≥ thr} equal `target`. Returns (P', edited mask, certificate)."""
    Q, edited, log = P.copy(), np.zeros(P.shape, bool), []
    for it in range(max_iter):
        b = betti_2d(Q >= thr); log.append(b)
        if b == tuple(target):
            break
        if b[0] > target[0]:                                    # too many components: remove the weakest surplus ones
            lab, nc = ndimage.label(Q >= thr, structure=CONN8)
            mass = ndimage.sum(Q, lab, index=np.arange(1, nc + 1))
            for l in np.argsort(mass)[: nc - target[0]] + 1:
                sel = lab == l
                Q[sel] = np.minimum(Q[sel], thr - margin); edited |= sel
        elif b[1] < target[1]:                                  # a hole is missing: close the cheapest gap
            fg, ends, best = Q >= thr, skeleton_endpoints(Q >= thr), None
            for i in range(len(ends)):
                for j in range(i + 1, len(ends)):
                    if np.linalg.norm(ends[i] - ends[j]) > 20:
                        continue
                    cost = np.where(fg, 10.0, np.maximum(thr + margin - Q, 0) + 1e-3)   # do not walk around the ring
                    cost[tuple(ends[i])] = cost[tuple(ends[j])] = 1e-3
                    path, c = route_through_array(cost, tuple(ends[i]), tuple(ends[j]), fully_connected=True)
                    if best is None or c < best[0]:
                        best = (c, path)
            if best is None:
                break
            for y, x in best[1]:
                if Q[y, x] < thr + margin:
                    Q[y, x] = thr + margin; edited[y, x] = True
        else:
            break
    final = betti_2d(Q >= thr)
    cert = dict(iterations=len(log) - (final == tuple(target)), edited_fraction=float(edited.mean()),
                converged=final == tuple(target), betti_path=log + [final])
    return Q, edited, cert


Q, edited, cert = topo_repair(P)
print("certificate:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in cert.items()})
print(f"Dice before {dice(P >= 0.5, gt_ring):.3f} → after {dice(Q >= 0.5, gt_ring):.3f}; "
      f"Betti before {betti_2d(P >= 0.5)} → after {betti_2d(Q >= 0.5)} (target (1, 1))")
assert cert["converged"] and cert["edited_fraction"] < 0.02, "the repair should converge with sparse edits"

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
axes[0].imshow(P, cmap="viridis"); axes[0].set_title("probability map")
axes[1].imshow(P >= 0.5); axes[1].set_title(f"threshold 0.5: β = {betti_2d(P >= 0.5)}")
axes[2].imshow(Q >= 0.5); axes[2].contour(edited, [0.5], colors="red", linewidths=1)
axes[2].set_title(f"repaired: β = {betti_2d(Q >= 0.5)}; red = edited pixels")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Two iterations and a handful of pixels (well under 1 % of the image — that number *is* the
certificate) turned an open vesicle with two ghosts into one closed vesicle. The edits are **local and auditable**: you
can show a reviewer exactly which pixels were changed, which is far more honest than a network that silently
"hallucinates" a closed membrane. ⚠️ The repair is only as good as its **target**: told that the cell holds one vesicle,
it will delete a real second one. TopoFuse therefore *predicts* the target Betti numbers from the image with a small
"topology prior" head, and flags the cases where the projection does not converge.

### 5.5 Tri-planar fusion: three 2D views of one 3D object

Foundation models such as SAM are 2D. TopoFuse (and, judging from its title, MVGFormer) process a volume as three stacks
of 2D slices — xy, xz and yz — and **fuse** the three predictions per voxel. Different views make different mistakes, so
fusion can vote them away. TopoFuse goes further: a small 3D network predicts per-voxel plane weights
$\alpha^{(p)}(\mathbf x)$, modulated (FiLM: feature-wise scale and shift) by a **topology descriptor** of the prediction,

$$Z(\mathbf x)=\sum_{p\in\{xy,xz,yz\}}\alpha^{(p)}(\mathbf x)\,Z^{(p)}(\mathbf x),\qquad \sum_p\alpha^{(p)}(\mathbf x)=1,$$

*in words:* each voxel's final score is a weighted vote of the three views, and the weights depend on where you are and
on how the topology currently looks. We test the simplest version (equal weights) on two vesicles in a 48³-nm volume
with a ±60° missing wedge and noise. Our per-view "segmenter" smooths each 2D slice and thresholds it.

In [ ]:
# ── 5.5 Tri-planar segmentation and fusion of two vesicles ──
g3 = np.mgrid[0:48, 0:48, 0:48].astype(float)
d_a = np.sqrt((g3[0] - 24) ** 2 + (g3[1] - 18) ** 2 + (g3[2] - 17) ** 2)
d_b = np.sqrt((g3[0] - 24) ** 2 + (g3[1] - 33) ** 2 + (g3[2] - 34) ** 2)
vesicles = (np.abs(d_a - 11) < 1.5) | (np.abs(d_b - 7) < 1.5)
vol3 = normalize(simulate_tomogram(ndimage.gaussian_filter(vesicles.astype(np.float32), 1.0),
                                   np.random.default_rng(4), snr=0.5, tilt_max_deg=60, ctf_pixel_A=None))
probs = {}
for ax, name in [(0, "xy"), (1, "xz"), (2, "yz")]:
    sig = [1.5, 1.5, 1.5]; sig[ax] = 0                          # 2D processing within each slice of this view
    sm = normalize(ndimage.gaussian_filter(vol3, sig))
    probs[name] = 1 / (1 + np.exp(-3 * (sm - 1.5)))             # a per-view 'probability of membrane'
probs["fused (mean of 3)"] = np.mean([probs[k] for k in ("xy", "xz", "yz")], axis=0)
print(f"truth: Betti (β0, β2) = {betti_3d(vesicles)}  (two closed vesicles)")
for name, p in probs.items():
    m = p >= 0.5
    print(f"  {name:18s} Dice {dice(m, vesicles):.2f}   Betti (β0, β2) = {betti_3d(m)}")
fused = probs["fused (mean of 3)"] >= 0.5
lab3, n3 = ndimage.label(fused, structure=np.ones((3, 3, 3), int))
sizes = ndimage.sum(fused, lab3, index=np.arange(1, n3 + 1))
keep = np.isin(lab3, np.argsort(-sizes)[:2] + 1)                   # target β0 = 2: drop the surplus small pieces
print(f"  fused + β0 repair  Dice {dice(keep, vesicles):.2f}   Betti (β0, β2) = {betti_3d(keep)}   "
      f"(edited {int((fused & ~keep).sum())} of {fused.size:,} voxels)")

fig, axes = plt.subplots(2, 4, figsize=(11, 5.4))
for j, name in enumerate(["xy", "xz", "yz", "fused (mean of 3)"]):
    axes[0, j].imshow(probs[name][24], cmap="viridis", vmin=0, vmax=1)
    axes[0, j].set_title(f"{name} view\ntop view: xy slice at z = 24 nm", fontsize=8)
    axes[1, j].imshow(probs[name][:, 18, :], cmap="viridis", vmin=0, vmax=1)
    axes[1, j].set_title(f"{name} view\nside view: xz slice at y = 18 nm", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Per-view membrane probabilities and their fusion (bottom row: side view, beam vertical)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each view alone produced many spurious fragments (β₀ far above 2); averaging the three views
removed most of them while keeping the Dice of the best single view — views disagree on noise but agree on structure. Yet **no** version
recovers β₂ = 2: in the side view (bottom row) the tops and bottoms of both vesicles are missing. Those membrane patches
are perpendicular to the beam, and their information lies inside the **missing wedge** (Module 07) — *every* view lacks
it, so voting cannot bring it back. Closing those caps requires a prior about shape — exactly what a topology target
plus a projection such as ProjT (which in TopoFuse acts on β₀ and β₂) is designed to supply, with the edits reported in
the certificate.

✏️ **Your turn.** In cell 5.5 change `tilt_max_deg=60` to `90` (no missing wedge) and re-run. Does β₂ come back? Then
try `snr=0.2`. Which Betti number is most sensitive to noise, and which to the missing wedge?

<details><summary>Hint</summary>Without a missing wedge the caps are measured, so the fused segmentation can close and
β₂ can reach its true value (fragments may still need the β₀ clean-up). Lower SNR mostly inflates β₀ (more fragments);
the missing wedge mostly destroys β₂ (open caps).</details>

### 📄 Paper connection — TopoFuse

**Paper**: Salla RK, Gupta N, Li X, Xu M. *TopoFuse: Topology-Aware Tri-Planar Fusion for 3D Cryo-Electron Tomography
Segmentation.* ECCV 2026 (poster; [PDF](https://media.eventhosts.cc/Conferences/ECCV2026/pdfs/4700.pdf)). Code and the
synthetic SYN dataset: [github.com/rohitsalla/TopoFuse](https://github.com/rohitsalla/TopoFuse) (MIT licence; not part of AITom).

- **Question it answers:** voxel-accurate segmentations are often topologically wrong (broken membranes, merged
  organelles, collapsed cavities) and topology *losses* cannot guarantee a fix — can we enforce topology with
  guarantees and an audit trail?
- **What the paper actually does:** (1) a shared **SAM ViT-B** encoder (first 8 blocks frozen, last 4 fine-tuned) with a
  2D decoder segments xy, xz and yz slice stacks; a topology descriptor (feature counts at persistence 0.05 and 0.15 plus
  total persistence) **FiLM-modulates** a 3D CNN that predicts per-voxel plane weights $\alpha^{(p)}(\mathbf x)$ (softmax
  over planes). (2) **ProjT**: topology as a projection — sparse edits $\Delta Z_c(\mathbf x)=-\eta\nabla d_B$ only at
  critical birth/death voxels of unmatched persistence pairs (cubical persistent homology on a 2× downsampled grid, for
  $d\in\{0,2\}$, i.e. components and cavities; tunnels excluded), at most 5 iterations with line search, and a
  **certificate** (iterations, edited-voxel fraction, convergence flag). (3) A **topology prior head** predicts the
  target $(\hat\beta_0,\hat\beta_2)$ at inference. Loss: Dice+CE on the projected output (+ 0.5 × Dice+CE on the raw
  output) + $0.1\,\mathcal L_{topo}$ (Wasserstein distance between persistence diagrams) + $0.05\,\mathcal L_{prior}$.
- **What our toy keeps:** Betti numbers via dual connectivity, persistence by union–find, a target-driven sparse repair
  with a certificate, tri-planar fusion.
- **What our toy leaves out:** SAM features and learning, FiLM-conditioned weights, the bottleneck-distance gradient
  step and line search (we use greedy discrete edits), the prior head, and 3D cavity repair.
- **Result reported:** against the strongest soft-loss baseline (BettiMatching), the authors report that the β₀ error
  falls by 54 % (2.68 → 1.24) and mean Dice rises from 80.3 to 84.9, editing a median 3.1 % of voxels; about 4 % of
  volumes do not converge (flagged by the certificate). Note that the gain over the *plain* tri-planar backbone is larger
  (+8.7 to +10.4 Dice), and the text contains some internal numeric inconsistencies.

> 🧐 **Critical-reading exercise.** TopoFuse describes EMPIAR-10499 as "HeLa cell lamellae". The lab's own CryoSAM and
> TomoPicker papers (and DISCA) describe EMPIAR-10499 as *Mycoplasma pneumoniae* cells. One of them must be wrong — most
> likely the TopoFuse description. Check the EMPIAR entry yourself; then ask: does a mislabelled dataset change any
> conclusion of the paper? (Its description of EMPIAR-10045 as a "bacterial tomogram" also needs checking: that entry is
> commonly known as a purified yeast 80S ribosome dataset.)

### 📄 Paper connection — MVGFormer (citation only)

**Paper**: Li H, Li X, Wang H, Shi J, Chen H, Zhao Y, Du B, Barthélemy J, Kihara D, Shen J, Xu M. *MVGFormer: Multi-view
perspective with graph-guided transformer for cryo-ET segmentation.* Knowledge-Based Systems 331:114810 (2025).
[doi:10.1016/j.knosys.2025.114810](https://doi.org/10.1016/j.knosys.2025.114810)

> ⚠️ We could not access the full text of this paper while writing; the description below is limited to its title and
> citation. Read the paper before relying on details.

- **What we can say:** the title indicates a *multi-view* treatment of tomograms for segmentation — the same broad idea
  as the tri-planar views of Section 5.5 — combined with a graph-guided transformer. We do not describe its
  architecture, data or results here because we have not verified them. The author list overlaps with Vox-UDA
  (Module 20).

---
## 6 · The big picture: how much supervision does each tool need?

| Method (module) | Human input | Training? | Output | Main risk |
|---|---|---|---|---|
| Template matching / DoG (10) | a template, or a size | no | particle positions | template bias; misses unknowns |
| Supervised 3D segmentation (16) | dense voxel masks | yes | voxel labels | label cost; domain shift |
| Vox-UDA (18) | labels on *simulated* data only | yes | voxel labels on real data | simulation gap |
| TomoPicker (23) | ~10 clicked particles | yes (PU learning) | particle positions | class-prior estimate |
| **CryoSAM (24)** | **1 click** (or a few) | **no** | 3D masks + positions | features must separate classes |
| **SD segmentation (24)** | **none** (choose slabs) | only the fast pseudo-label models | membranes + particle positions | heuristics; systematic pseudo-label errors |
| **TopoFuse (24)** | dense masks (training) | yes | topology-corrected masks + certificate | wrong topology target |

💡 Reading tip: whenever a paper claims "few" or "no" labels, ask *where the knowledge comes from instead* — a pretrained
foundation model, a simulator, a prior on shape — and what happens when that source is wrong for your cells.

---
## ✅ Checkpoint

1. Why does CryoSAM stop propagating a mask when the IoU between neighbouring slices drops below 0.5?
   <details><summary>Answer</summary>A sudden loss of overlap means the self-prompt has slipped off the particle (onto a
   neighbour or the background) or the particle has ended; continuing would grow the mask into the wrong object.</details>
2. What does the coarse-to-fine pyramid with pruning at 0.5 buy, and what can it cost?
   <details><summary>Answer</summary>Speed: most of the tomogram is discarded at a cheap coarse level, so only a fraction
   of voxels is evaluated at full resolution. Cost: a true particle whose pooled coarse feature falls below 0.5 is pruned
   early and can never be recovered (lower recall).</details>
3. A row-stochastic matrix always has eigenvalue 1. Which eigenvector, and why is it useless for segmentation?
   <details><summary>Answer</summary>The constant vector (every row sums to 1, so A·1 = 1). It is the same everywhere, so
   it separates nothing; the informative eigenvectors are the next ones with eigenvalues close to 1.</details>
4. A predicted membrane has Dice 0.97 against the truth, but β₀ = 3 instead of 1. Give two biological misreadings this
   could cause.
   <details><summary>Answer</summary>Counting three organelles/vesicles instead of one; concluding that a membrane is
   ruptured or fragmented (e.g. a "damaged" mitochondrion); wrong volume or connectivity measurements per compartment.</details>
5. Why can't averaging the xy, xz and yz predictions restore the top of a vesicle in a ±60° tomogram?
   <details><summary>Answer</summary>The cap's Fourier components lie in the missing wedge, which is absent from the
   data itself; all three views are computed from the same data, so they all lack it. Only a prior (shape/topology)
   can fill it in, and that is an assumption that must be reported.</details>
6. What does a TopoFuse "certificate" contain, and why is it useful to a reader?
   <details><summary>Answer</summary>The number of iterations, the fraction of voxels edited and whether the projection
   converged. It shows how much of the final segmentation was imposed by the topology prior rather than seen in the data,
   and flags failures.</details>

## 🧠 Questions & discussion

1. CryoSAM is "training-free" — but SAM and DINOv2 were trained on natural photographs. In what sense is the method
   really label-free, and what could go wrong on a tomogram that looks nothing like a photograph?
2. Our toy found that the centroid query worked better than the masked average. Design an experiment that would tell
   you which query is better for *real* DINOv2 features.
3. The SD-segmentation pipeline labels round things as macromolecules. Which real cellular structures would break this
   rule (think: filaments in cross-section, small vesicles, lipid droplets)?
4. Pseudo-label training made our fast model *better* than its teacher. When would it make it *worse*?
5. Who should choose the target topology (β₀, β₁, β₂) of a segmentation — the biologist, a prior network, or the data?
   What should a paper report so that readers can judge?
6. The TopoFuse dataset description conflicts with the lab's other papers. How should you, as a reader, handle such a
   discrepancy when you cite or reuse a dataset?

## 🚀 Bonus — going further

- **Real foundation models.** On a Colab GPU, install `segment-anything` and run the official CryoSAM code in
  [AITom](https://github.com/xulabs/aitom/tree/master/aitom/segmentation/foundation_model/cryosam) on one denoised
  tomogram from EMPIAR-10499 (*M. pneumoniae*) or the [CZ cryoET Data Portal](https://cryoetdataportal.czscience.com/).
- **Better toy features.** Replace the filter bank in Section 3 by the activations of a small random 3D CNN (or the CNN
  you trained in Module 16). Do F1 and the paper-vs-code query comparison change?
- **True 3D topology.** Extend `topo_repair` to 3D cavities (β₂) for the vesicles of Section 5.5: detect the holes in the
  shell and close them with the cheapest surface patch. Report the certificate.
- **Persistence libraries.** Compare our union–find diagrams with `gudhi` or `cripser` (CubicalRipser, used by TopoFuse).
- **Deep dive.** The repo's [`TomogramSegmentation`](../../foundation/TomogramSegmentation/) tutorials cover supervised membrane
  segmentation with real data.

## 📝 Lab note

```text
Module 32 — date:
Input: in silico cell (ribosomes, proteasomes, RuBisCO, vesicle; SNR 0.1, ±60°); 2D membrane slabs; ring and vesicle phantoms
Output: one-click detections (P/R/F1), zero-label membrane/particle masks (Dice/F1), Betti numbers, repaired masks + certificate
Method: tri-view filter bank + whitening; self-prompting (IoU < 0.5 stop); pruned coarse-to-fine cosine matching; NMS;
        attention eigenvectors (eig vs |zᵀAz−1| loss); circularity split; union–find persistence; sparse repair; tri-planar fusion
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: CryoSAM (MICCAI 2024); Uddin et al. (CVPR 2026); TopoFuse (ECCV 2026); MVGFormer (KBS 2025, unread)
```

## 📚 References

1. Zhao Y, Bian H, Mu M, Uddin MR, Li Z, Li X, Wang T, Xu M. CryoSAM: Training-free CryoET tomogram segmentation with foundation models. *MICCAI 2024*, LNCS, pp. 124–134. doi:10.1007/978-3-031-72111-3_12 (arXiv:2407.06833)
2. Uddin MR, Tabib HMS, Nguyen T-H, Gandhi K, Xu M. Unsupervised multi-scale segmentation of 3D subcellular world with Stable Diffusion foundation model. *CVPR 2026*, pp. 22744–22752.
3. Salla RK, Gupta N, Li X, Xu M. TopoFuse: Topology-aware tri-planar fusion for 3D cryo-electron tomography segmentation. *ECCV 2026*.
4. Li H, Li X, Wang H, Shi J, Chen H, Zhao Y, Du B, Barthélemy J, Kihara D, Shen J, Xu M. MVGFormer: Multi-view perspective with graph-guided transformer for cryo-ET segmentation. *Knowledge-Based Systems* 331:114810 (2025). doi:10.1016/j.knosys.2025.114810
5. Kirillov A *et al.* Segment Anything. *ICCV 2023*. arXiv:2304.02643
6. Oquab M *et al.* DINOv2: Learning robust visual features without supervision. *TMLR* (2024). arXiv:2304.07193
7. Rombach R, Blattmann A, Lorenz D, Esser P, Ommer B. High-resolution image synthesis with latent diffusion models. *CVPR 2022*. arXiv:2112.10752
8. Shi J, Malik J. Normalized cuts and image segmentation. *IEEE TPAMI* 22:888–905 (2000). doi:10.1109/34.868688
9. Edelsbrunner H, Harer J. *Computational Topology: An Introduction.* American Mathematical Society (2010).
10. Uddin MR *et al.* Localization of macromolecules in crowded cellular cryo-electron tomograms from extremely sparse labels (TomoPicker). *Brief Bioinform* 26:bbaf630 (2025). doi:10.1093/bib/bbaf630

## ➡️ Next

**[33 · From maps to biology: membranes, drugs, fitting atoms and the proteome in situ](33_from_maps_to_biology.ipynb)** —
you will turn segmentations and particle maps into biological measurements, using a real study of how an antifungal drug
reshapes the fungal plasma membrane. Course overview: [`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).